# PTK Metrics Comparison Plotting


Plotting interface appears at very bottom (run all cells).

---
*Annette Thompson · Fox and Shirts Labs, CU Boulder · 2026 · Developed with assistance from GitHub Copilot (Claude Sonnet 4.6)*

In [1]:
# Imports & Configuration

import warnings, re, threading
import concurrent.futures as _cf
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.axes import Axes as _MplAxes
from scipy.ndimage import gaussian_filter1d

RESULTS_ROOT = Path("./results")


In [2]:
# Data helpers

_CONFIG: dict = {}
_DF_CACHE: dict[tuple, pd.DataFrame | None] = {}
_META_CACHE: dict[str, dict[str, dict]] = {}
_RUNS_CACHE: dict[tuple[str, str], tuple[str, ...]] = {}
_CACHE_LOCK = threading.Lock()
_SCORE_COLS: list[str] = []
_DEFAULT_MODULES: list[str] = []


def _discover_state_cols(score_cols: list[str], root: Path = RESULTS_ROOT) -> list[str]:
    """Columns present in metrics.csv but absent from metadata and not score/time cols."""
    _time = {"frame", "time", "time_ps", "time_ns"}
    _scores = set(score_cols)
    if not root.is_dir():
        return []
    for sys_dir in sorted(root.iterdir()):
        if not sys_dir.is_dir():
            continue
        for run_dir in sorted(sys_dir.iterdir()):
            meta_path = run_dir / "Active_Metrics" / "metrics_metadata.csv"
            csv_path  = run_dir / "Active_Metrics" / "metrics.csv"
            if not meta_path.exists() or not csv_path.exists():
                continue
            try:
                meta_cols = set(
                    pd.read_csv(meta_path, usecols=["output_column"])["output_column"].astype(str)
                )
                csv_cols = set(pd.read_csv(csv_path, nrows=0).columns)
                found = sorted(csv_cols - meta_cols - _scores - _time)
                if found:
                    return found
            except Exception:
                continue
    return []


_STATE_COLS: list[str] = _discover_state_cols(_SCORE_COLS) or [
    "atp_transfer_ready", "mg_catalytic", "a_loop_stabilized", "peptide_ready",
]
_DERIVED_COLS = set(_SCORE_COLS + _STATE_COLS)
_SKIP_COLS = {"frame", "time", "time_ps", "time_ns", *_DERIVED_COLS}


# RDF center labels are discovered from RDF filenames.

_RDF_LABEL_OVERRIDES: dict[str, str] = {
    "mg":                  "Mg",
    "pep_Y":               "Pep Y",
    "cat_arg":             "Cat Arg",
    "hrd_asp":             "HRD Asp",
    "atp_gamma_phosphate": "ATP gamma phosphate",
}


def _rdf_label_for(sel_a: str) -> str:
    if sel_a in _RDF_LABEL_OVERRIDES:
        return _RDF_LABEL_OVERRIDES[sel_a]
    _upper = {"atp", "hrd", "dfg", "ptk", "ac"}
    parts = sel_a.split("_")
    return " ".join(p.upper() if p in _upper else p.capitalize() for p in parts)


_RDF_CENTER_LABELS: dict[str, str] = {}


# Preferred display order only; actual RDF type options come from RDF filenames.
_RDF_TYPE_ORDER: list[str] = ["all", "water", "overall_first_shell", "ptk", "peptide", "atp", "mg"]


In [3]:
# Raw Metrics Data Loading

def _load_df(system: str, run: str, root: Path = RESULTS_ROOT) -> pd.DataFrame | None:
    key = (str(root), system, run)
    with _CACHE_LOCK:
        if key in _DF_CACHE:
            return _DF_CACHE[key]
    csv_path = root / system / run / "Active_Metrics" / "metrics.csv"
    if not csv_path.exists():
        with _CACHE_LOCK:
            _DF_CACHE[key] = None
        return None
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        warnings.warn(f"Could not read {csv_path}: {exc}")
        df = None
    with _CACHE_LOCK:
        _DF_CACHE[key] = df
    return df


def discover_systems(root: Path = RESULTS_ROOT) -> list[str]:
    systems = []
    if root.is_dir():
        systems = sorted(
            d.name for d in root.iterdir()
            if d.is_dir()
            and list(d.glob("*/Active_Metrics/metrics.csv"))
        )
    return sorted(systems)


def discover_runs(system: str, root: Path = RESULTS_ROOT) -> list[str]:
    key = (str(root), system)
    if key not in _RUNS_CACHE:
        sys_dir = root / system
        if not sys_dir.is_dir():
            runs: tuple[str, ...] = tuple()
        else:
            runs = tuple(sorted(
                d.name for d in sys_dir.iterdir()
                if d.is_dir()
                and (d / "Active_Metrics" / "metrics.csv").exists()
            ))
        _RUNS_CACHE[key] = runs
    return list(_RUNS_CACHE[key])


def prefetch(systems: list[str], runs: list[str] | None = None, root: Path = RESULTS_ROOT) -> None:
    tasks = []
    for system in systems:
        for run in (runs or discover_runs(system, root)):
            key = (str(root), system, run)
            with _CACHE_LOCK:
                already = key in _DF_CACHE
            if not already:
                tasks.append((system, run))
    if not tasks:
        return
    with _cf.ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(lambda t: _load_df(t[0], t[1], root), tasks))


def load_system_metadata(system: str, root: Path = RESULTS_ROOT) -> dict[str, dict]:
    for run in discover_runs(system, root):
        meta_path = root / system / run / "Active_Metrics" / "metrics_metadata.csv"
        if not meta_path.exists():
            continue
        try:
            df = pd.read_csv(meta_path)
            return {str(row["output_column"]): row.to_dict() for _, row in df.iterrows()}
        except Exception:
            continue
    return {}


def _get_meta(system: str, root: Path = RESULTS_ROOT) -> dict[str, dict]:
    if system not in _META_CACHE:
        _META_CACHE[system] = load_system_metadata(system, root)
    return _META_CACHE[system]


def _get_shared_meta(systems: list[str], root: Path = RESULTS_ROOT) -> dict[str, dict]:
    for system in systems:
        meta = _get_meta(system, root)
        if meta:
            return meta
    return {}


def _is_hbond_metric(metric: str) -> bool:
    return "hbond" in metric


def _is_hidden_metric(metric: str) -> bool:
    return metric in _SKIP_COLS or metric in _DERIVED_COLS


def _visible_metric_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if not _is_hidden_metric(c)]


def load_run_means(system: str, run: str, root: Path = RESULTS_ROOT) -> dict[str, float] | None:
    df = _load_df(system, run, root)
    if df is None:
        return None
    means: dict[str, float] = {}
    for col in _visible_metric_columns(df):
        s = pd.to_numeric(df[col], errors="coerce")
        if not s.notna().any():
            continue
        val = float(s.mean(skipna=True))
        if _is_hbond_metric(col):
            val *= 100.0
        means[col] = val
    return means


def collect_stats(systems: list[str], runs: list[str] | None = None, root: Path = RESULTS_ROOT) -> dict[str, dict[str, list[float]]]:
    prefetch(systems, runs, root)
    result: dict[str, dict[str, list[float]]] = {}
    for system in systems:
        run_means = []
        for run in (runs or discover_runs(system, root)):
            means = load_run_means(system, run, root)
            if means is not None:
                run_means.append((run, means))
        if not run_means:
            continue
        keys = set().union(*[set(m.keys()) for _, m in run_means])
        result[system] = {k: [m[k] for _, m in run_means if k in m] for k in keys}
    return result


def collect_individual(systems: list[str], runs: list[str] | None = None, root: Path = RESULTS_ROOT) -> list[tuple[str, str, dict[str, float]]]:
    prefetch(systems, runs, root)
    rows: list[tuple[str, str, dict[str, float]]] = []
    for system in systems:
        for run in (runs or discover_runs(system, root)):
            means = load_run_means(system, run, root)
            if means is not None:
                rows.append((system, run, means))
    return rows


def load_run_raw(system: str, run: str, metric: str, root: Path = RESULTS_ROOT) -> np.ndarray | None:
    df = _load_df(system, run, root)
    if df is None or _is_hidden_metric(metric):
        return None
    if metric not in df.columns:
        return None
    s = pd.to_numeric(df[metric], errors="coerce").dropna().to_numpy(dtype=float)
    return s if len(s) else None


def mean_sem(values: list[float]) -> tuple[float, float]:
    arr = np.array(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan, np.nan
    mean = float(arr.mean())
    sem = float(arr.std(ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
    return mean, sem


def all_metric_names(stats_data: dict) -> list[str]:
    if not stats_data:
        return []
    return sorted(set().union(*[set(v.keys()) for v in stats_data.values()]))


In [4]:
# RDF File Discovery

def _parse_rdf_source(source: str) -> dict:
    parts = source.split("_") if source else []
    charge_index = next((i for i, part in enumerate(parts) if part in {"pos", "neg"}), None)
    if charge_index is None:
        type_parts = parts or ["all"]
        charge = ""
        label_parts: list[str] = []
    else:
        type_parts = parts[:charge_index] or ["all"]
        charge = parts[charge_index]
        label_parts = parts[charge_index + 1:]
    rdf_type = "_".join(type_parts)
    label = "_".join(label_parts)
    return {"type": rdf_type, "charge": charge, "label": label, "is_detail": bool(label)}


def _split_rdf_stem(stem: str) -> tuple[str, str] | None:
    marker = "_from_coordination_"
    if marker not in stem:
        return None
    center_base, source = stem.split(marker, 1)
    center = f"{center_base}_from_coordination"
    _RDF_CENTER_LABELS.setdefault(center, _rdf_label_for(center_base))
    return center, source


def discover_rdf_records(system: str, run: str, root: Path = RESULTS_ROOT) -> list[dict]:
    rdf_dir = root / system / run / "Active_Metrics"
    if not rdf_dir.is_dir():
        return []
    records: list[dict] = []
    for path in sorted(rdf_dir.glob("rdf_*.csv")):
        stem = path.stem[4:]
        split = _split_rdf_stem(stem)
        if split is None:
            continue
        center, source = split
        info = _parse_rdf_source(source)
        records.append({
            "key": f"{center}:{source}",
            "center": center,
            "source": source,
            "type": info["type"],
            "charge": info["charge"],
            "label": info["label"],
            "is_detail": info["is_detail"],
        })
    return records


def discover_rdf_catalog(systems: list[str], root: Path = RESULTS_ROOT) -> list[dict]:
    seen: set[str] = set()
    catalog: list[dict] = []
    for system in systems:
        for run in discover_runs(system, root):
            for rec in discover_rdf_records(system, run, root):
                if rec["key"] not in seen:
                    seen.add(rec["key"])
                    catalog.append(rec)
    return catalog


def _rdf_record_display(rec: dict) -> str:
    center = _RDF_CENTER_LABELS.get(rec["center"], rec["center"].replace("_", " ").title())
    bits = [rec["type"].upper() if rec["type"] in {"atp", "ptk"} else rec["type"].title()]
    if rec.get("charge"):
        bits.append(rec["charge"])
    if rec.get("label"):
        bits.append(rec["label"].replace("_", " "))
    return f"{center} - {' · '.join(bits)}"


def _rdf_matches_type(rec: dict, rdf_type: str, detail: bool) -> bool:
    if detail != bool(rec.get("is_detail")):
        return False
    return rdf_type == "all" or rec.get("type") == rdf_type


def load_rdf_series(system: str, run: str, record_key: str, root: Path = RESULTS_ROOT) -> pd.DataFrame | None:
    center, source = record_key.split(":", 1)
    path = root / system / run / "Active_Metrics" / f"rdf_{center}_{source}.csv"
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if not {"r_ang", "g_rdf"}.issubset(df.columns):
        return None
    return df[["r_ang", "g_rdf"]]


In [5]:
# Module and Threshold Semantics

_RDF_CHARGE_GROUP_PREFIX = "__rdf_charge_group__::"
_RDF_CHARGE_CONTACT_RE = re.compile(
    r"^(?P<center>.+_from_coordination)_(?P<source>.+?)_(?P<charge>pos|neg)(?:_(?P<detail>.+))?_(?P<op>dist|hbond(?:_.+)?)$",
    re.IGNORECASE,
)

def _is_rdf_charge_group_value(value: str | None) -> bool:
    return isinstance(value, str) and value.startswith(_RDF_CHARGE_GROUP_PREFIX)


def _rdf_charge_key_from_value(value: str) -> str:
    return value[len(_RDF_CHARGE_GROUP_PREFIX):]


def _mtype(col: str, meta: dict | None = None) -> str:
    if callable(globals().get("_is_rdf_charge_group_value")) and _is_rdf_charge_group_value(col):
        key = _rdf_charge_key_from_value(col)
        return "hbond" if key.endswith("|hbond") else "distance"
    if meta and col in meta:
        row = meta[col]
        op = str(row.get("operation") or "").lower()
        units = str(row.get("units") or "").lower()
        kind = str(row.get("output_kind") or "").lower()
        if "hbond" in op or "hbond" in kind:
            return "hbond"
        if "coord" in op or "coord" in kind or "_cn" in kind:
            return "coordination"
        if "angle" in op or "deg" in units:
            return "angle"
        if "dist" in op or "angstrom" in units or "å" in units:
            return "distance"
    if _is_hbond_metric(col):
        return "hbond"
    if col.endswith("_angle"):
        return "angle"
    if "_cn_" in col or col.endswith("_cn") or "from_coordination_cn" in col:
        return "coordination"
    if col in _STATE_COLS:
        return "binary"
    return "distance"


def _resolve_metric_names(cfg: dict):
    seen: dict[str, int] = {}
    for mdef in (dict(m) for m in cfg.get("metrics", [])):
        if mdef.get("name") and mdef.get("name") != "default":
            name = mdef["name"]
        else:
            sel = mdef.get("selection")
            sel_a = mdef.get("selection_a")
            sel_b = mdef.get("selection_b")
            base = sel or (f"{sel_a}_{sel_b}" if sel_a and sel_b else sel_a) or "metric"
            seen[base] = seen.get(base, 0) + 1
            name = base if seen[base] == 1 else f"{base}_{seen[base]}"
        mdef["name"] = name
        yield name, mdef


def _valid_score_name(value) -> bool:
    return bool(value) and value not in ("null", "false", False, None)


def _build_score_group_map(cfg: dict) -> dict[str, str]:
    result: dict[str, str] = {}
    for name, mdef in _resolve_metric_names(cfg):
        ops = mdef.get("ops") or {}
        expand_a = bool(mdef.get("expand_a"))
        expand_b = bool(mdef.get("expand_b"))
        full_contact_map = bool(mdef.get("full_contact_map"))
        for op_key, op_val in ops.items():
            if not isinstance(op_val, dict):
                continue
            if op_key == "coordination":
                for entry in op_val.get("track", []):
                    if not isinstance(entry, dict):
                        continue
                    score = entry.get("score")
                    group = entry.get("group")
                    if _valid_score_name(score) and group:
                        result.setdefault(f"{name}_cn_{group}", score)
                        result.setdefault(f"{name}_{group}", score)
                continue
            score = op_val.get("score")
            if not _valid_score_name(score):
                continue
            result.setdefault(name, score)
            if op_key == "distance":
                result.setdefault(f"{name}_dist", score)
            elif op_key == "angle":
                result.setdefault(f"{name}_angle", score)
            elif op_key == "hbond":
                result.setdefault(f"{name}_hbond", score)
    return result


def _get_module_names_from_config(cfg: dict) -> list[str]:
    names = []
    for item in cfg.get("scoring", []):
        if isinstance(item, dict) and "combine_groups" not in item and item.get("name"):
            names.append(item["name"])
    return names


def _parse_threshold(value) -> tuple[float | None, float | None]:
    if value in (None, "", "special"):
        return None, None
    if not isinstance(value, (list, tuple)):
        value = [value]
    def _f(item):
        try:
            if item in (None, "", "nan", "NaN") or pd.isna(item):
                return None
            return float(item)
        except (ValueError, TypeError):
            return None
    lo = _f(value[0]) if len(value) >= 1 else None
    hi = _f(value[1]) if len(value) >= 2 else None
    return lo, hi


def _build_threshold_map(cfg: dict) -> dict[str, tuple[float | None, float | None]]:
    result: dict[str, tuple[float | None, float | None]] = {}
    for name, mdef in _resolve_metric_names(cfg):
        ops = mdef.get("ops") or {}
        expand_b = bool(mdef.get("expand_b"))
        full_contact_map = bool(mdef.get("full_contact_map"))
        for op_key, op_val in ops.items():
            if not isinstance(op_val, dict):
                continue
            raw_threshold = op_val.get("threshold", mdef.get("threshold"))
            lo, hi = _parse_threshold(raw_threshold)
            if lo is None and hi is None:
                continue
            if op_key == "distance":
                result.setdefault(f"{name}_dist", (lo, hi))
            elif op_key == "angle":
                result.setdefault(f"{name}_angle", (lo, hi))
            elif op_key == "hbond":
                result.setdefault(f"{name}_hbond", (lo, hi))
            elif op_key == "coordination":
                for entry in op_val.get("track", []):
                    if not isinstance(entry, dict):
                        continue
                    group = entry.get("group")
                    entry_lo, entry_hi = _parse_threshold(entry.get("threshold"))
                    if group and not (entry_lo is None and entry_hi is None):
                        result.setdefault(f"{name}_cn_{group}", (entry_lo, entry_hi))
    return result


_SCORE_GROUP_MAP = _build_score_group_map(_CONFIG)
_CONFIG_THRESHOLD_MAP = _build_threshold_map(_CONFIG)
_MODULE_NAMES = _get_module_names_from_config(_CONFIG) or list(_DEFAULT_MODULES)


_OP_MARKERS = ("_dist", "_angle", "_hbond", "_cn")


def _split_base_and_op(col: str) -> tuple[str, str]:
    # returns (base_metric_name, operation)
    best_idx = None
    best_op = ""
    for marker in _OP_MARKERS:
        idx = col.find(marker)
        if idx != -1 and (best_idx is None or idx < best_idx):
            best_idx = idx
            best_op = marker[1:]
    if best_idx is None:
        return col, ""
    return col[:best_idx], best_op

def _scoring_group_of(col: str) -> str:
    if col in _SCORE_GROUP_MAP:
        return _SCORE_GROUP_MAP[col]
    base, _op = _split_base_and_op(col)
    if base in _SCORE_GROUP_MAP:
        return _SCORE_GROUP_MAP[base]
    best = ""
    for key in _SCORE_GROUP_MAP:
        if col.startswith(key + "_") and len(key) > len(best):
            best = key
    return _SCORE_GROUP_MAP.get(best, "")

_MTYPE_UNIT = {
    "distance": "A",
    "angle": "deg",
    "hbond": "%",
    "coordination": "CN",
    "binary": "0-1",
}
_MTYPE_YLABEL = {
    "distance": "Distance (Å)",
    "angle": "Angle (deg)",
    "hbond": "% Contact",
    "coordination": "Coordination Number",
    "binary": "Fraction (0-1)",
}


def _thresh_for(col: str, meta: dict | None = None) -> tuple[float | None, float | None]:
    if meta and col in meta:
        def _f(value):
            try:
                if value in (None, "", "nan", "NaN") or pd.isna(value):
                    return None
                return float(value)
            except (ValueError, TypeError):
                return None
        lo, hi = _f(meta[col].get("threshold_lo")), _f(meta[col].get("threshold_hi"))
        if lo is not None or hi is not None:
            return lo, hi
    return _CONFIG_THRESHOLD_MAP.get(col, (None, None))


def _add_thresh_marker(ax: plt.Axes, lo: float | None, hi: float | None, orientation: str = "v") -> None:
    span_fn = ax.axvspan if orientation == "v" else ax.axhspan
    line_fn = ax.axvline if orientation == "v" else ax.axhline
    if lo is not None and hi is not None:
        span_fn(lo, hi, alpha=0.10, color="limegreen", zorder=0)
    if lo is not None:
        line_fn(lo, ls="--", lw=1, color="limegreen", alpha=0.7, zorder=1)
    if hi is not None:
        line_fn(hi, ls="--", lw=1, color="limegreen", alpha=0.7, zorder=1)


In [6]:
# Shared Dashboard Widgets

CUSTOM_PALETTES = {
    "Custom 4-Color": ["#50C878", "#4A90E2", "#FFC107", "#17A2B8"],
}


_ALL_SYSTEMS = discover_systems(RESULTS_ROOT)
_DEFAULT_SELECTED_SYSTEMS = []
_SYS_DEFAULTS_APPLIED = False


def _checkbox_panel(title: str):
    btn_all = widgets.Button(
        description="All",
        layout=widgets.Layout(width="60px", min_width="60px")
    )
    btn_none = widgets.Button(
        description="Clear",
        layout=widgets.Layout(width="60px", min_width="60px")
    )
    search = widgets.Text(
        placeholder=f"Filter {title.lower()}...",
        layout=widgets.Layout(width="160px", min_width="160px")
    )
    controls = widgets.HBox(
        [btn_all, btn_none, search],
        layout=widgets.Layout(gap="6px", align_items="center")
    )
    grid = widgets.VBox([])
    panel = widgets.VBox([controls, grid])
    return panel, search, btn_all, btn_none


_sys_panel, _sys_search, _sys_btn_all, _sys_btn_none = _checkbox_panel("Systems")
_sys_checks: dict[str, widgets.Checkbox] = {}
_sys_all_rows: list = []
_system_bulk_timer: threading.Timer | None = None
_system_update_timer = globals().get("_system_update_timer", None)


def _rebuild_sys_checkboxes(*_):
    global _sys_all_rows
    global _SYS_DEFAULTS_APPLIED
    prev = {s for s, cb in _sys_checks.items() if cb.value}
    selected = set(prev)
    if not _SYS_DEFAULTS_APPLIED and not selected:
        selected = set(_DEFAULT_SELECTED_SYSTEMS) & set(_ALL_SYSTEMS)
        _SYS_DEFAULTS_APPLIED = True
    _sys_checks.clear()
    _sys_all_rows = []
    rows = []
    for system in _ALL_SYSTEMS:
        cb = widgets.Checkbox(description=system, value=(system in selected), indent=False)
        _sys_checks[system] = cb
        _sys_all_rows.append((system, cb))
        rows.append(cb)
    _sys_panel.children[1].children = rows


def _filter_sys(*_):
    text_value = (_sys_search.value or "").strip().lower()
    _sys_panel.children[1].children = [cb for name, cb in _sys_all_rows if not text_value or text_value in name.lower()]


def _get_selected_systems() -> list[str]:
    return [s for s, cb in _sys_checks.items() if cb.value]


def _step_system_values(target_value: bool, delay: float = 0.02):
    global _system_bulk_timer
    pending = [cb for cb in _sys_checks.values() if cb.value != target_value]
    if not pending:
        return

    def _apply_next(index: int = 0):
        global _system_bulk_timer
        if index >= len(pending):
            _system_bulk_timer = None
            return
        pending[index].value = target_value
        _system_bulk_timer = threading.Timer(delay, lambda: _apply_next(index + 1))
        _system_bulk_timer.daemon = True
        _system_bulk_timer.start()

    if _system_bulk_timer is not None:
        _system_bulk_timer.cancel()
    _apply_next(0)


def _select_all_systems(*_):
    _step_system_values(True)


def _clear_all_systems(*_):
    _step_system_values(False)


_sys_btn_all.on_click(_select_all_systems)
_sys_btn_none.on_click(_clear_all_systems)
_rebuild_sys_checkboxes()
_run_mode = widgets.ToggleButtons(options=["Averaged", "Individual runs"], value="Averaged")


def _is_individual() -> bool:
    return _run_mode.value == "Individual runs"


_color_pickers: dict[str, widgets.ColorPicker] = {}
_color_box = widgets.VBox([])
_UNIT_COLOR_MAP: dict[str, str] = {}


_THEMES = ["Default", "Seaborn - White", "Seaborn - Whitegrid", "Seaborn - Dark", "Seaborn - Darkgrid", "Seaborn - Ticks", "Matplotlib - ggplot", "Matplotlib - fivethirtyeight", "Matplotlib - bmh", "Matplotlib - classic"]
_PALETTES = ["Default", "Custom 4-Color", "hls", "husl", "tab10", "tab20", "colorblind", "deep", "muted", "bright", "dark", "pastel", "Set1", "Set2", "Set3", "Dark2", "Paired", "Accent", "coolwarm", "viridis", "plasma", "magma", "Blues", "Greens", "Purples", "Reds"]
_theme_dd      = widgets.Dropdown(options=_THEMES, value="Default", description="Theme:")
_palette_dd    = widgets.Dropdown(options=_PALETTES, value="Custom 4-Color", description="Palette:")
_error_mode_dd = widgets.Dropdown(options=["SEM", "95% CI"], value="SEM", description="Error bars:")
_dots_cb       = widgets.Checkbox(value=True, description="Show run dots")
_ts_smooth     = widgets.BoundedIntText(value=1, min=1, max=501, step=2, description="Smooth:")
_font_size     = widgets.BoundedIntText(value=10, min=6, max=36, description="Font size:")
_font_fam      = widgets.Dropdown(options=["sans-serif", "serif", "DejaVu Sans", "Arial", "Helvetica", "Times New Roman"], value="Arial", description="Font:")
_dpi_it        = widgets.BoundedIntText(value=300, min=72, max=600, description="DPI:")
_fig_auto      = widgets.Checkbox(value=True, description="Auto figure size")
_fig_w         = widgets.BoundedFloatText(value=4.0, min=1.0, max=30.0, step=0.5, description="Plot W (in):")
_fig_h         = widgets.BoundedFloatText(value=2.5, min=1.0, max=30.0, step=0.5, description="Plot H (in):")


def _palette_colors(systems: list[str], palette: str) -> dict[str, str]:
    pal = CUSTOM_PALETTES.get(palette, None if palette == "Default" else palette)
    colors = sns.color_palette(pal, n_colors=max(1, len(systems)))
    return {s: mcolors.to_hex(colors[i]) for i, s in enumerate(systems)}


def _apply_font_settings():
    plt.rcParams.update({"font.size": _font_size.value, "axes.titlesize": _font_size.value, "axes.labelsize": _font_size.value, "xtick.labelsize": _font_size.value, "ytick.labelsize": _font_size.value, "legend.fontsize": _font_size.value, "figure.titlesize": _font_size.value, "font.family": _font_fam.value, "figure.dpi": _dpi_it.value, "savefig.dpi": _dpi_it.value})


def _apply_theme(theme: str, palette: str) -> None:
    sns.reset_defaults()
    plt.style.use("default")
    if theme.startswith("Seaborn - "):
        sns.set_theme(style=theme[10:].lower().replace(" ", "_"))
    elif theme.startswith("Matplotlib - "):
        plt.style.use(theme[13:].lower())
    pal_obj = CUSTOM_PALETTES.get(palette, None if palette == "Default" else palette)
    if pal_obj is not None:
        sns.set_palette(pal_obj)
    if callable(globals().get("_apply_font_settings")):
        _apply_font_settings()


def _rebuild_color_pickers(*_):
    systems = _get_selected_systems()
    if not systems:
        _color_box.children = [widgets.HTML("<span style='font-size:11px;color:#888'>Select at least one system to show color pickers.</span>")]
        return
    defaults = _palette_colors(systems, _palette_dd.value)
    _color_pickers.clear()
    pickers = []
    for system in systems:
        color = _UNIT_COLOR_MAP.get(system, defaults[system])
        _UNIT_COLOR_MAP[system] = color
        cp = widgets.ColorPicker(
            description=system,
            value=color,
            style={"description_width": "160px"},
            layout=widgets.Layout(width="280px", min_width="280px")
        )
        def _on_color(change, _s=system):
            if change["name"] == "value":
                _UNIT_COLOR_MAP[_s] = change["new"]
        cp.observe(_on_color, names="value")
        _color_pickers[system] = cp
        pickers.append(cp)
    _color_box.children = [widgets.VBox(pickers)]


def _get_color_map(systems: list[str]) -> dict[str, str]:
    defaults = _palette_colors(systems, _palette_dd.value)
    return {s: _UNIT_COLOR_MAP.get(s, defaults[s]) for s in systems}


def _get_fig_size() -> tuple[float, float] | None:
    return None if _fig_auto.value else (_fig_w.value, _fig_h.value)


_use_xlim_cb = widgets.Checkbox(value=False, description="Custom X limits")
_xmin_ft     = widgets.FloatText(value=0.0, description="X min:")
_xmax_ft     = widgets.FloatText(value=1.0, description="X max:")
_use_ylim_cb = widgets.Checkbox(value=False, description="Custom Y limits")
_ymin_ft     = widgets.FloatText(value=0.0, description="Y min:")
_ymax_ft     = widgets.FloatText(value=1.0, description="Y max:")
_title_wt    = widgets.Text(value="", description="Title:")
_xlabel_wt   = widgets.Text(value="", description="X label:")
_ylabel_wt   = widgets.Text(value="", description="Y label:")
_xlim_box = widgets.HBox([_xmin_ft, _xmax_ft])
_ylim_box = widgets.HBox([_ymin_ft, _ymax_ft])


def _get_axis_overrides():
    xlim = (_xmin_ft.value if _use_xlim_cb.value else None, _xmax_ft.value if _use_xlim_cb.value else None)
    ylim = (_ymin_ft.value if _use_ylim_cb.value else None, _ymax_ft.value if _use_ylim_cb.value else None)
    title = (_title_wt.value or "").strip() or None
    xlabel = (_xlabel_wt.value or "").strip() or None
    ylabel = (_ylabel_wt.value or "").strip() or None
    return xlim, ylim, title, xlabel, ylabel


def _sync_axis_boxes(*_):
    _xlim_box.layout.display = "" if _use_xlim_cb.value else "none"
    _ylim_box.layout.display = "" if _use_ylim_cb.value else "none"


_sync_axis_boxes()


_CONTACT_MAP_MODULE_LABEL = "PTK-peptide contact map"
_CONTACT_MAP_CATEGORY     = "ptk_peptide_contact_map"
_MODULE_OPTIONS = ["(all metrics)", _CONTACT_MAP_MODULE_LABEL] + _MODULE_NAMES + ["coordination", "unscored / other"]
_MTYPE_FILTER_MAP = {"All types": None, "Distance (A)": "distance", "Angle (deg)": "angle", "H-Bond (%)": "hbond", "Coordination (CN)": "coordination"}
_THRESHOLD_CHART_LABEL = "Threshold %"
_RMS_CATALOG: list[dict] = []
_METADATA_SEARCH_FIELDS = (
    "output_column", "display_name", "plot_label", "search_text", "base_metric_name",
    "operation", "output_kind", "units", "rdf_derived", "rdf_source",
    "entity_a", "entity_b", "entity_a_type", "entity_b_type",
    "generic_label_a", "generic_label_b", "specific_label_a", "specific_label_b",
    "position_label_a", "position_label_b", "display_label_a", "display_label_b",
    "group_a", "group_b", "group_a_residues", "group_b_residues", "group_a_atoms", "group_b_atoms",
)
_CONTACT_OCC_CACHE: dict[tuple[str, str, str, str, float], float | None] = globals().get("_CONTACT_OCC_CACHE", {})
_RDF_CATALOG: list[dict] = []

_mtype_filter_dd  = widgets.Dropdown(options=list(_MTYPE_FILTER_MAP.keys()), value="All types", description="Type:")
_metric_dd        = widgets.Dropdown(options=[], description="Metric:")
_chart_type = widgets.ToggleButtons(
    options=["Bar (mean)", "KDE", "Time Series", _THRESHOLD_CHART_LABEL],
    value="Bar (mean)",
)
_show_thresh_cb    = widgets.Checkbox(value=False, description="Show threshold shading")
_thresh_lo_wt      = widgets.BoundedFloatText(value=0.0, min=-9999.0, max=9999.0, step=0.1, description="Lo:")
_thresh_hi_wt      = widgets.BoundedFloatText(value=0.0, min=-9999.0, max=9999.0, step=0.1, description="Hi:")
_thresh_override_box = widgets.HBox([_thresh_lo_wt, _thresh_hi_wt])
_chart_sub_box     = widgets.VBox([])
_plot_btn = widgets.Button(description="Plot", button_style="success")
_save_btn = widgets.Button(description="Save PNG")
_status   = widgets.HTML("")
_out      = widgets.Output()
_rms_kind_dd = widgets.Dropdown(options=[("RMSD", "rmsd"), ("RMSF", "rmsf")], value="rmsd", description="Kind:")
_rms_metric_dd = widgets.Dropdown(options=[], description="Metric:")
_rms_smooth = widgets.BoundedIntText(value=1, min=1, max=101, step=2, description="Smooth win:")
_rms_msg = widgets.HTML("")
_contact_filter_cb = widgets.Checkbox(value=False, description="Filter options by distance occupancy")
_contact_cutoff_wt = widgets.BoundedFloatText(
    value=6.0,
    min=0.0,
    max=50.0,
    step=0.1,
    description="Cutoff (A):",
)
_contact_min_pct_wt = widgets.BoundedFloatText(
    value=10.0,
    min=0.0,
    max=100.0,
    step=1.0,
    description="Min traj (%):",
)
_contact_filter_msg = widgets.HTML("<span style='font-size:11px;color:#aaa'>Passes when at least one selected run is below cutoff for the requested trajectory fraction.</span>")
_metrics_help = widgets.HTML("<span style='font-size:11px;color:#777'>Choose a metric type and metric, then plot averaged or individual runs.</span>")
_rdf_help = widgets.HTML("<span style='font-size:11px;color:#777'>Choose center/type, optionally residues, then plot RDF curves across selected systems.</span>")
_rms_help = widgets.HTML("<span style='font-size:11px;color:#777'>Choose RMS kind and metric, then pick bar/KDE/series options.</span>")

_metrics_pane = widgets.VBox([
    _metrics_help,
    _mtype_filter_dd,
    _metric_dd,
])
_rdf_center_dd = widgets.Dropdown(options=[], description="Center:")
_rdf_type_dd = widgets.Dropdown(options=[], description="Type:")
_rdf_view = widgets.ToggleButtons(options=["Aggregate", "Individual residues"], value="Aggregate")
_rdf_detail_dd = widgets.SelectMultiple(options=[], description="Residues:", rows=8, layout=widgets.Layout(width="90%"))
_rdf_msg = widgets.HTML("")
_rdf_pane = widgets.VBox([
    _rdf_help,
    _rdf_center_dd, _rdf_type_dd, _rdf_view, _rdf_detail_dd, _rdf_msg,
])
_rms_pane = widgets.VBox([
    _rms_help,
    _rms_kind_dd, _rms_metric_dd,
    _rms_msg,
])

In [7]:
# Metric Search and Metric Dropdown

_MT_SORT_ORDER = {"distance": 0, "angle": 1, "hbond": 2, "coordination": 3, "binary": 4}
_metric_update_timer: threading.Timer | None = None


def _is_coord_metric(metric: str) -> bool:
    return "_cn_" in metric or metric.endswith("_cn") or "from_coordination_cn" in metric


def _meta_row_for_metric(metric: str, meta: dict | None = None) -> dict:
    return (meta or {}).get(metric, {}) or {}


def _metadata_value_text(value) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value)


def _clean_metric_label_text(value: str) -> str:
    text_value = _metadata_value_text(value)
    text_value = text_value.replace("_from_coordination", "")
    text_value = text_value.replace("_From Coordination", "")
    text_value = text_value.replace("_neg", "")
    text_value = text_value.replace("_pos", "")
    return text_value


def _metric_search_text(metric: str, meta: dict | None = None, mode: str | None = None) -> str:
    return str(metric).lower()


def _metric_dropdown_label(metric: str, meta: dict | None = None) -> str:
    return str(metric)


def _get_effective_threshold() -> tuple[float | None, float | None]:
    return float(_thresh_lo_wt.value), float(_thresh_hi_wt.value)


In [8]:
# RDF Charge Group Metrics

def _rdf_charge_match(metric: str) -> re.Match | None:
    return _RDF_CHARGE_CONTACT_RE.match(str(metric))


def _is_rdf_derived_metric(metric: str, meta: dict | None = None) -> bool:
    row = _meta_row_for_metric(metric, meta)
    flag = _metadata_value_text(row.get("rdf_derived")).strip().lower()
    if flag in {"true", "1", "yes", "y"}:
        return True
    if _metadata_value_text(row.get("rdf_source")).strip():
        return True
    return "from_coordination" in metric


def _is_rdf_distance_or_hbond_metric(metric: str, meta: dict | None = None) -> bool:
    if not _is_rdf_derived_metric(metric, meta):
        return False
    return _mtype(metric, meta) in {"distance", "hbond"}


def _rdf_charge_group_key(metric: str, meta: dict | None = None) -> str | None:
    if not _is_rdf_distance_or_hbond_metric(metric, meta):
        return None
    matched = _rdf_charge_match(metric)
    if matched is None:
        return None
    detail = matched.group("detail") or "all"
    if detail != "all":
        return None
    op = "hbond" if "hbond" in matched.group("op").lower() else "dist"
    return "|".join([matched.group("center"), matched.group("source"), "aggregate", op])


def _rdf_charge_group_value(key: str) -> str:
    return f"{_RDF_CHARGE_GROUP_PREFIX}{key}"


def _rdf_charge_group_label(key: str) -> str:
    center, source, detail_level, op = key.split("|", 3)
    center_label = _RDF_CENTER_LABELS.get(center, center.replace("_from_coordination", "").replace("_", " ").title())
    source_label = source.upper() if source in {"ptk", "atp"} else source.title()
    detail_label = "charged residues" if detail_level == "residue" else "charged aggregate"
    op_label = "H-bond" if op == "hbond" else "distance"
    return f"{center_label} - {source_label} {detail_label} {op_label}"


def _rdf_charge_metric_detail_label(metric: str, meta: dict | None = None) -> str:
    matched = _rdf_charge_match(metric)
    if matched is None:
        return _metric_dropdown_label(metric, meta) if callable(globals().get("_metric_dropdown_label")) else metric
    charge = matched.group("charge").lower()
    detail = matched.group("detail") or "all"
    detail = detail[:-5] if detail.endswith("_dist") else detail
    detail = detail[:-6] if detail.endswith("_hbond") else detail
    charge_label = "pos" if charge == "pos" else "neg"
    return f"{charge_label}: {detail.replace('_', ' ')}"


def _rdf_charge_groups_from_metrics(metrics: list[str], meta: dict | None = None) -> dict[str, list[str]]:
    groups: dict[str, list[str]] = defaultdict(list)
    for metric in metrics:
        key = _rdf_charge_group_key(metric, meta)
        if key is not None:
            groups[key].append(metric)
    usable: dict[str, list[str]] = {}
    for key, members in groups.items():
        charges = {(_rdf_charge_match(metric).group("charge") or "").lower() for metric in members if _rdf_charge_match(metric)}
        if len(members) >= 2 and {"pos", "neg"}.issubset(charges):
            usable[key] = sorted(members, key=lambda metric: (_rdf_charge_metric_detail_label(metric, meta), metric))
    return usable


def _rdf_charge_group_members(value: str, systems: list[str], meta: dict | None = None) -> list[str]:
    key = _rdf_charge_key_from_value(value)
    groups = _rdf_charge_groups_from_metrics(_all_current_metric_columns(systems), meta)
    return groups.get(key, [])


def _rdf_charge_group_title(value: str) -> str:
    return _rdf_charge_group_label(_rdf_charge_key_from_value(value))


def _combine_rdf_charge_arrays(arrays: list[np.ndarray], mt: str) -> np.ndarray | None:
    clean = [np.asarray(arr, dtype=float) for arr in arrays if arr is not None and len(arr)]
    if not clean:
        return None
    n = min(len(arr) for arr in clean)
    if n == 0:
        return None
    mat = np.stack([arr[:n] for arr in clean])
    if mt == "hbond":
        return np.nanmax(mat, axis=0)
    return np.nanmin(mat, axis=0)


def _load_rdf_charge_group_raw(system: str, run: str, metrics: list[str], mt: str, root: Path = RESULTS_ROOT) -> np.ndarray | None:
    arrays: list[np.ndarray] = []
    for metric in metrics:
        arr = load_run_raw(system, run, metric, root)
        if arr is not None:
            arrays.append(arr)
    return _combine_rdf_charge_arrays(arrays, mt)


def _rdf_charge_group_run_means(metrics: list[str], systems: list[str], mt: str, root: Path = RESULTS_ROOT) -> dict[str, dict[str, list[float]]]:
    result: dict[str, dict[str, list[float]]] = {}
    for system in systems:
        values: list[float] = []
        for run in discover_runs(system, root):
            combined = _load_rdf_charge_group_raw(system, run, metrics, mt, root)
            if combined is None:
                continue
            value = float(np.nanmean(combined))
            if mt == "hbond":
                value *= 100.0
            values.append(value)
        if values:
            result[system] = {"combined": values}
    return result


def _rdf_charge_group_individual_means(metrics: list[str], systems: list[str], mt: str, root: Path = RESULTS_ROOT) -> list[tuple[str, str, dict[str, float]]]:
    rows: list[tuple[str, str, dict[str, float]]] = []
    for system in systems:
        for run in discover_runs(system, root):
            combined = _load_rdf_charge_group_raw(system, run, metrics, mt, root)
            if combined is None:
                continue
            value = float(np.nanmean(combined))
            if mt == "hbond":
                value *= 100.0
            rows.append((system, run, {"combined": value}))
    return rows


def _plot_rdf_charge_group_bar(metrics: list[str], systems: list[str], color_map: dict[str, str], mt: str,
                               individual: bool = False, root: Path = RESULTS_ROOT, title: str | None = None,
                               xlabel: str | None = None, ylabel: str | None = None, show_dots: bool = True,
                               error_mode: str = "SEM", xlim=None, ylim=None, fig_size: tuple | None = None) -> plt.Figure:
    if individual:
        data = _rdf_charge_group_individual_means(metrics, systems, mt, root)
        return plot_bar_individual("combined", data, color_map, title=title, xlabel=xlabel, ylabel=ylabel,
                                   xlim=xlim, ylim=ylim, fig_size=fig_size)
    stats_data = _rdf_charge_group_run_means(metrics, systems, mt, root)
    return plot_bar("combined", systems, stats_data, color_map, title=title, xlabel=xlabel, ylabel=ylabel,
                    show_dots=show_dots, error_mode=error_mode, xlim=xlim, ylim=ylim, fig_size=fig_size)


def _plot_rdf_charge_group_kde(metrics: list[str], systems: list[str], color_map: dict[str, str], mt: str,
                               individual: bool = False, root: Path = RESULTS_ROOT, title: str | None = None,
                               xlabel: str | None = None, ylabel: str | None = None, xlim=None, ylim=None,
                               fig_size: tuple | None = None) -> plt.Figure:
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    for system in systems:
        runs = discover_runs(system, root)
        arrays: list[tuple[str, np.ndarray]] = []
        for run in runs:
            arr = _load_rdf_charge_group_raw(system, run, metrics, mt, root)
            if arr is not None:
                arrays.append((run, arr[np.isfinite(arr)]))
        if not arrays:
            continue
        color = color_map.get(system, "#888")
        if individual:
            shades = _run_shades(system, [run for run, _ in arrays], color)
            for run, arr in arrays:
                if len(arr) >= 5:
                    sns.kdeplot(arr, ax=ax, color=shades.get(run, color), label=f"{system} / {run}", lw=1, fill=True, alpha=0.08)
        else:
            combined = np.concatenate([arr for _, arr in arrays])
            combined = combined[np.isfinite(combined)]
            if len(combined) >= 5:
                sns.kdeplot(combined, ax=ax, color=color, label=system, lw=1, fill=True, alpha=0.15)
    ax.set_xlabel(xlabel or "Distance (Å)")
    ax.set_ylabel(ylabel or "Density")
    ax.set_title(title or "RDF first-shell charge contacts - Distribution")
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def _plot_rdf_charge_group_timeseries(metrics: list[str], systems: list[str], color_map: dict[str, str], mt: str,
                                      individual: bool = False, root: Path = RESULTS_ROOT, error_mode: str = "SEM",
                                      smooth_window: int = 1, title: str | None = None, xlabel: str | None = None,
                                      ylabel: str | None = None, xlim=None, ylim=None, fig_size: tuple | None = None) -> plt.Figure:
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    for system in systems:
        runs = discover_runs(system, root)
        arrays: list[tuple[str, np.ndarray]] = []
        for run in runs:
            arr = _load_rdf_charge_group_raw(system, run, metrics, mt, root)
            if arr is not None:
                if mt == "hbond":
                    arr = arr * 100.0
                arrays.append((run, arr))
        if not arrays:
            continue
        color = color_map.get(system, "#888")
        if individual:
            shades = _run_shades(system, [run for run, _ in arrays], color)
            for run, arr in arrays:
                y = _smooth_series_values(arr, smooth_window)
                ax.plot(np.arange(len(y)), y, lw=1, color=shades.get(run, color), label=f"{system} / {run}", alpha=0.85)
        else:
            n = min(len(arr) for _, arr in arrays)
            mat = np.stack([arr[:n] for _, arr in arrays])
            mean_arr = _smooth_series_values(np.nanmean(mat, axis=0), smooth_window)
            err_arr = (np.array([_ci95(mat[:, j]) for j in range(n)]) if error_mode == "95% CI" and mat.shape[0] > 1
                       else (np.nanstd(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0]) if mat.shape[0] > 1 else np.zeros(n)))
            err_arr = _smooth_series_values(err_arr, smooth_window)
            frames = np.arange(len(mean_arr))
            ax.plot(frames, mean_arr, lw=1, color=color, label=system)
            ax.fill_between(frames, mean_arr - err_arr, mean_arr + err_arr, color=color, alpha=0.20)
    ax.set_xlabel(xlabel or "Frame")
    ax.set_ylabel(ylabel or ("% frames with H-bond" if mt == "hbond" else "Distance (Å)"))
    ax.set_title(title or "RDF first-shell charge contacts - Time Series")
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def _build_metric_options(systems: list[str], mtype_sel: str) -> list[tuple[str, str]]:
    if not systems:
        return []
    meta = _get_shared_meta(systems)
    metrics = _all_current_metric_columns(systems) if callable(globals().get("_all_current_metric_columns")) else all_metric_names(collect_stats(systems, None, RESULTS_ROOT))
    filtered = list(metrics)
    want = _MTYPE_FILTER_MAP.get(mtype_sel)
    if want is not None:
        filtered = [m for m in filtered if _mtype(m, meta) == want]
    groups = _rdf_charge_groups_from_metrics(filtered, meta) if callable(globals().get("_rdf_charge_groups_from_metrics")) else {}
    collapsed: list[tuple[str, str]] = []
    emitted: set[str] = set()
    for metric in sorted(filtered, key=lambda m: (_MT_SORT_ORDER.get(_mtype(m, meta), 99), _metric_dropdown_label(m, meta).lower(), m)):
        group_key = _rdf_charge_group_key(metric, meta) if callable(globals().get("_rdf_charge_group_key")) else None
        if group_key in groups:
            if group_key not in emitted:
                collapsed.append(("  " + _rdf_charge_group_label(group_key), _rdf_charge_group_value(group_key)))
                emitted.add(group_key)
            continue
        collapsed.append(("  " + _metric_dropdown_label(metric, meta), metric))
    return collapsed


def _update_metric_options(*_):
    systems = _get_selected_systems()
    previous = _metric_dd.value
    opts = _build_metric_options(systems, _mtype_filter_dd.value)
    _metric_dd.options = opts
    if not opts:
        return
    values = [value for _, value in opts]
    _metric_dd.value = previous if previous in values else values[0]


def _schedule_metric_update(*_) -> None:
    global _metric_update_timer
    if _metric_update_timer is not None:
        _metric_update_timer.cancel()
    _metric_update_timer = threading.Timer(0.35, _update_metric_options)
    _metric_update_timer.start()



In [9]:
# Co-occurrence Metrics

_DEFAULT_COOCCURRENCE_LOGIC = "all"
_DEFAULT_COOCCURRENCE_MIN_COUNT = 3
_DEFAULT_COOCCURRENCE_ROWS = [{'use': True, 'metric': 'pep_Y_atp_2_angle', 'lo': '120', 'hi': '180'}, {'use': True, 'metric': 'aC_glu_b3_lys_dist', 'lo': '0', 'hi': '4'}, {'use': True, 'metric': 'cat_arg_from_coordination_cn_atp_neg', 'lo': '1', 'hi': ''}, {'use': True, 'metric': 'pep_Y_hrd_asp_dist', 'lo': '0', 'hi': '4'}]
_COOCCURRENCE_TYPE_OPTIONS = [("All types", ""), ("Distance", "distance"), ("H-Bond", "hbond"), ("Angle", "angle"), ("Coordination", "coordination"), ("Binary", "binary")]
_cooccurrence_rows: list[dict[str, object]] = []

_COOCCURRENCE_PANEL_WIDTH = "700px"
_COOCCURRENCE_ROW_TOP_WIDTH = "690px"
_COOCCURRENCE_ROW_BOTTOM_WIDTH = "300px"

_cooccurrence_logic = widgets.ToggleButtons(
    options=[("All conditions", "all"), ("Any condition", "any"), ("At least N", "at_least")],
    value=_DEFAULT_COOCCURRENCE_LOGIC,
    description="Logic:",
    layout=widgets.Layout(width="360px", min_width="360px"),
)
_cooccurrence_logic.style.button_width = "108px"
_cooccurrence_min_count = widgets.BoundedIntText(
    value=_DEFAULT_COOCCURRENCE_MIN_COUNT,
    min=1,
    max=24,
    step=1,
    description="N:",
    layout=widgets.Layout(width="160px", min_width="160px"),
)
_cooccurrence_num_metrics = widgets.BoundedIntText(
    value=4,
    min=1,
    max=24,
    step=1,
    description="Metrics:",
    layout=widgets.Layout(width="190px", min_width="190px"),
)
_cooccurrence_help = widgets.HTML("<span style='font-size:11px;color:#777'>Define metric bounds and logic to measure co-occurrence across trajectory frames.</span>")
_cooccurrence_info = widgets.HTML("")
_cooccurrence_rows_box = widgets.VBox([], layout=widgets.Layout(width=_COOCCURRENCE_PANEL_WIDTH, gap="4px"))
_cooccurrence_header_controls = widgets.HBox(
    [_cooccurrence_logic, _cooccurrence_min_count, _cooccurrence_num_metrics],
    layout=widgets.Layout(width=_COOCCURRENCE_PANEL_WIDTH, flex_flow="row wrap", gap="8px", align_items="center"),
)
_cooccurrence_pane = widgets.VBox([
    _cooccurrence_help,
    _cooccurrence_header_controls,
    _cooccurrence_rows_box,
    _cooccurrence_info,
], layout=widgets.Layout(width=_COOCCURRENCE_PANEL_WIDTH))


def _cooccurrence_metric_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=[],
        description="Metric:",
        style={"description_width": "50px"},
        layout=widgets.Layout(width="450px", min_width="450px"),
    )


def _cooccurrence_type_dropdown(value: str = "") -> widgets.Dropdown:
    return widgets.Dropdown(
        options=_COOCCURRENCE_TYPE_OPTIONS,
        value=value if value in dict(_COOCCURRENCE_TYPE_OPTIONS).values() else "",
        description="Type:",
        style={"description_width": "42px"},
        layout=widgets.Layout(width="145px", min_width="145px"),
    )


def _cooccurrence_bound_text(description: str) -> widgets.Text:
    return widgets.Text(
        value="",
        placeholder="optional",
        description=description,
        style={"description_width": "28px"},
        layout=widgets.Layout(width="145px", min_width="145px"),
    )


def _cooccurrence_metric_options(systems: list[str], search: str = "", mtype_filter: str | None = None) -> list[tuple[str, str]]:
    if not systems:
        return []
    meta = _get_shared_meta(systems)
    metrics = _all_current_metric_columns(systems) if callable(globals().get("_all_current_metric_columns")) else all_metric_names(collect_stats(systems, None, RESULTS_ROOT))
    options: list[tuple[str, str]] = []
    search_term = (search or "").strip().lower()
    for metric in sorted(metrics, key=lambda m: (_MT_SORT_ORDER.get(_mtype(m, meta), 99), _metric_dropdown_label(m, meta).lower(), m)):
        if callable(globals().get("_is_rdf_charge_group_value")) and _is_rdf_charge_group_value(metric):
            continue
        metric_type = _mtype(metric, meta)
        if mtype_filter and metric_type != mtype_filter:
            continue
        if search_term and search_term not in _metric_search_text(metric, meta):
            continue
        options.append((_metric_dropdown_label(metric, meta), metric))
    return options


def _cooccurrence_preferred_metrics(options: list[tuple[str, str]]) -> list[str | None]:
    values = [value for _, value in options]
    preferred = [
        "pep_Y_atp_2_angle",
        "aC_glu_b3_lys_dist",
        "cat_arg_from_coordination_cn_atp_neg",
        "pep_Y_hrd_asp_dist",
    ]
    chosen: list[str | None] = []
    for metric in preferred:
        chosen.append(metric if metric in values else None)
    return chosen


def _format_bound(value: float | None) -> str:
    return "" if value is None else f"{value:g}"


def _cooccurrence_default_bounds(metric: str | None, meta: dict | None = None) -> tuple[float | None, float | None]:
    if not metric:
        return None, None
    lo, hi = _thresh_for(metric, meta)
    if lo is not None or hi is not None:
        return lo, hi
    metric_type = _mtype(metric, meta)
    if metric_type == "hbond":
        return 0.5, None
    if metric_type == "binary":
        return 0.5, None
    return None, None


def _cooccurrence_row_default_bounds(row_index: int, metric: str | None, meta: dict | None = None) -> tuple[float | None, float | None]:
    if row_index < len(_DEFAULT_COOCCURRENCE_ROWS):
        default_row = _DEFAULT_COOCCURRENCE_ROWS[row_index]
        def _bound(value):
            value = str(value or "").strip()
            if not value:
                return None
            try:
                return float(value)
            except ValueError:
                return None
        lo = _bound(default_row.get("lo"))
        hi = _bound(default_row.get("hi"))
        if lo is not None or hi is not None:
            return lo, hi
    return _cooccurrence_default_bounds(metric, meta)


def _set_cooccurrence_row_bounds(row_index: int, metric: str | None) -> None:
    if row_index >= len(_cooccurrence_rows):
        return
    systems = _get_selected_systems()
    meta = _get_shared_meta(systems) if systems else {}
    lo, hi = _cooccurrence_row_default_bounds(row_index, metric, meta)
    _cooccurrence_rows[row_index]["lo"].value = _format_bound(lo)
    _cooccurrence_rows[row_index]["hi"].value = _format_bound(hi)


def _update_cooccurrence_metric_options(*_) -> None:
    systems = _get_selected_systems()
    if not systems:
        for row in _cooccurrence_rows:
            row["metric"].options = []
            row["metric"].value = None
        _cooccurrence_info.value = "<span style='color:orange'>Select at least one system to configure co-occurrence metrics.</span>"
        return
    search = ""
    options = _cooccurrence_metric_options(systems, search)
    preferred_metrics = _cooccurrence_preferred_metrics(options)
    value_set = {value for _, value in options}
    for index, row in enumerate(_cooccurrence_rows):
        metric_widget: widgets.Dropdown = row["metric"]
        type_widget: widgets.Dropdown = row["type"]
        mtype_filter = type_widget.value or None
        filtered_options = _cooccurrence_metric_options(systems, search, mtype_filter)
        metric_widget.options = filtered_options
        previous = metric_widget.value
        preferred = preferred_metrics[index] if index < len(preferred_metrics) else None
        filtered_values = {value for _, value in filtered_options}
        if previous in filtered_values:
            metric_widget.value = previous
        elif preferred in filtered_values:
            metric_widget.value = preferred
        elif filtered_options:
            metric_widget.value = filtered_options[0][1]
        else:
            metric_widget.value = None
        _set_cooccurrence_row_bounds(index, metric_widget.value)
    _update_cooccurrence_info()


def _parse_cooccurrence_bound(widget: widgets.Text) -> float | None:
    value = widget.value.strip()
    if not value:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def _cooccurrence_specs() -> list[dict[str, object]]:
    specs: list[dict[str, object]] = []
    max_rows = int(_cooccurrence_num_metrics.value)
    for index, row in enumerate(_cooccurrence_rows):
        if index >= max_rows:
            continue
        if not row["use"].value:
            continue
        metric = row["metric"].value
        if not metric:
            continue
        lo = _parse_cooccurrence_bound(row["lo"])
        hi = _parse_cooccurrence_bound(row["hi"])
        if lo is None and hi is None:
            continue
        specs.append({"metric": metric, "lo": lo, "hi": hi})
    return specs


def _cooccurrence_required_count(spec_count: int) -> int:
    logic = _cooccurrence_logic.value
    if logic == "all":
        return spec_count
    if logic == "any":
        return 1 if spec_count else 0
    return max(1, min(int(_cooccurrence_min_count.value), spec_count))


def _cooccurrence_logic_label(specs: list[dict[str, object]] | int | None = None) -> str:
    spec_count = len(specs) if isinstance(specs, list) else int(specs or 0)
    required = _cooccurrence_required_count(spec_count)
    if required <= 0:
        return "No active conditions"
    if required == 1 and spec_count > 1:
        return "Any condition"
    if required == spec_count:
        return "All conditions"
    return f"At least {required} of {spec_count}"


def _sync_cooccurrence_logic_controls(*_) -> None:
    _cooccurrence_min_count.layout.display = ""
    specs = _cooccurrence_specs()
    _cooccurrence_info.value = f"<span style='font-size:11px;color:#777'>Logic: {_cooccurrence_logic_label(specs)}</span>"
def _sync_cooccurrence_row_visibility(*_) -> None:
    max_rows = int(_cooccurrence_num_metrics.value)
    for index, row in enumerate(_cooccurrence_rows):
        row["box"].layout.display = "" if index < max_rows else "none"
    _update_cooccurrence_info()


def _cooccurrence_spec_label(spec: dict[str, object]) -> str:
    metric = str(spec["metric"])
    lo = spec.get("lo")
    hi = spec.get("hi")
    if lo is not None and hi is not None:
        bounds = f"[{lo:g}, {hi:g}]"
    elif lo is not None:
        bounds = f">= {lo:g}"
    elif hi is not None:
        bounds = f"<= {hi:g}"
    else:
        bounds = "(no bounds)"
    return f"{metric} {bounds}"


def _cooccurrence_title(specs: list[dict[str, object]]) -> str:
    return f"Co-occurrence ({_cooccurrence_logic_label(specs)})"


def _cooccurrence_slug() -> str:
    specs = _cooccurrence_specs()
    if not specs:
        return "empty"
    chunks = []
    for spec in specs:
        metric = str(spec["metric"]).replace(" ", "_")
        lo = "na" if spec.get("lo") is None else f"{spec['lo']:g}".replace(".", "p")
        hi = "na" if spec.get("hi") is None else f"{spec['hi']:g}".replace(".", "p")
        chunks.append(f"{metric}_{lo}_{hi}")
    return "__".join(chunks)[:180]


def _cooccurrence_pct_for_run(system: str, run: str, specs: list[dict[str, object]], root: Path = RESULTS_ROOT) -> float | None:
    if not specs:
        return None
    arrays: list[np.ndarray] = []
    for spec in specs:
        metric = str(spec["metric"])
        arr = load_run_raw(system, run, metric, root)
        if arr is None:
            return None
        arr = np.asarray(arr, dtype=float)
        arrays.append(arr)
    min_len = min(len(arr) for arr in arrays)
    if min_len == 0:
        return None
    masks: list[np.ndarray] = []
    for arr, spec in zip(arrays, specs):
        values = arr[:min_len]
        lo, hi = spec.get("lo"), spec.get("hi")
        mask = np.ones(min_len, dtype=bool)
        if lo is not None:
            mask &= values >= float(lo)
        if hi is not None:
            mask &= values <= float(hi)
        masks.append(mask)
    required = _cooccurrence_required_count(len(masks))
    if required <= 0:
        return None
    stacked = np.stack(masks, axis=0)
    count_hits = np.sum(stacked, axis=0)
    meets = count_hits >= required
    return float(np.mean(meets) * 100.0)


def _collect_cooccurrence_stats(systems: list[str], specs: list[dict[str, object]], root: Path = RESULTS_ROOT) -> dict[str, dict[str, list[float]]]:
    result: dict[str, dict[str, list[float]]] = {}
    for system in systems:
        vals: list[float] = []
        for run in discover_runs(system, root):
            pct = _cooccurrence_pct_for_run(system, run, specs, root)
            if pct is not None:
                vals.append(pct)
        if vals:
            result[system] = {"cooccurrence_pct": vals}
    return result


def _collect_cooccurrence_individual(systems: list[str], specs: list[dict[str, object]], root: Path = RESULTS_ROOT) -> list[tuple[str, str, dict[str, float]]]:
    rows: list[tuple[str, str, dict[str, float]]] = []
    for system in systems:
        for run in discover_runs(system, root):
            pct = _cooccurrence_pct_for_run(system, run, specs, root)
            if pct is not None:
                rows.append((system, run, {"cooccurrence_pct": pct}))
    return rows


def _cooccurrence_summary_df(stats_data: dict[str, dict[str, list[float]]]) -> pd.DataFrame:
    rows = []
    for system, data in stats_data.items():
        values = data.get("cooccurrence_pct", [])
        if not values:
            continue
        rows.append({
            "System": system,
            "Mean": float(np.mean(values)),
            "Std": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
            "N": len(values),
        })
    return pd.DataFrame(rows).sort_values("Mean", ascending=False) if rows else pd.DataFrame(columns=["System", "Mean", "Std", "N"])


def _run_cooccurrence_plot(*_) -> None:
    systems = _get_selected_systems()
    specs = _cooccurrence_specs()
    if not systems:
        with _out:
            clear_output(wait=True)
            print("Select at least one system.")
        return
    if not specs:
        with _out:
            clear_output(wait=True)
            print("Configure at least one co-occurrence metric with bounds.")
        return
    individual = _is_individual()
    xlim, ylim, title, xlabel, ylabel = _get_axis_overrides()
    _apply_theme(_theme_dd.value, _palette_dd.value)
    color_map = _get_color_map(systems)
    use_legend = False
    with _out:
        clear_output(wait=True)
        plt.close("all")
        default_title = _cooccurrence_title(specs)
        if individual:
            rows = _collect_cooccurrence_individual(systems, specs, RESULTS_ROOT)
            if not rows:
                print("No co-occurrence data found for selected systems/runs.")
                return
            fig = plot_bar_individual("cooccurrence_pct", rows, color_map, title=title or default_title, xlabel=xlabel, ylabel=ylabel or "% Trajectory", xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), use_legend=use_legend)
            _show_fig(fig)
        else:
            stats_data = _collect_cooccurrence_stats(systems, specs, RESULTS_ROOT)
            if not stats_data:
                print("No co-occurrence data found for selected systems/runs.")
                return
            fig = plot_bar("cooccurrence_pct", systems, stats_data, color_map, title=title or default_title, show_dots=_dots_cb.value, error_mode=_error_mode_dd.value, xlabel=xlabel, ylabel=ylabel or "% Trajectory", xlim=xlim, ylim=ylim, fig_size=_get_fig_size())
            _show_fig(fig)
            summary = _cooccurrence_summary_df(stats_data)
            if not summary.empty:
                display(summary.style.format({"Mean": "{:.2f}", "Std": "{:.2f}"}))


def _update_cooccurrence_info(*_) -> None:
    specs = _cooccurrence_specs()
    if not specs:
        _cooccurrence_info.value = "<span style='color:orange'>No active co-occurrence conditions.</span>"
        return
    logic_text = _cooccurrence_logic_label(specs)
    lines = "<br>".join(f"- {_cooccurrence_spec_label(spec)}" for spec in specs)
    _cooccurrence_info.value = f"<div style='font-size:11px;color:#888'><b>{logic_text}</b><br>{lines}</div>"


def _cooccurrence_metric_changed(change, row_index: int) -> None:
    if change.get("name") != "value":
        return
    _set_cooccurrence_row_bounds(row_index, change.get("new"))
    _update_cooccurrence_info()


_CHART_TOGGLE_HIDDEN_TABS = {1, 3, 4, 5}
_PLOT_OPTIONS_HIDDEN_TABS = {5}


def _chart_options_for_tab(idx: int):
    if idx == 1:
        return ["KDE", "Time Series"]
    if idx == 2:
        series_label = "Atom Series" if getattr(_rms_kind_dd, "value", "rmsd") == "rmsf" else "Time Series"
        return [("Bar (mean)", "Bar (mean)"), ("KDE", "KDE"), (series_label, "Time Series")]
    if idx in (3, 4, 5):
        return ["Bar (mean)"]
    return ["Bar (mean)", "KDE", "Time Series", _THRESHOLD_CHART_LABEL]


def _option_values(options) -> list:
    return [opt[1] if isinstance(opt, tuple) else opt for opt in options]


def _sync_plot_option_visibility(idx: int) -> None:
    for widget_name, hidden_tabs in {
        "_chart_type_row": _CHART_TOGGLE_HIDDEN_TABS,
        "_section2_box": _PLOT_OPTIONS_HIDDEN_TABS,
        "_run_mode_block": _PLOT_OPTIONS_HIDDEN_TABS,
    }.items():
        widget = globals().get(widget_name)
        if widget is not None:
            widget.layout.display = "none" if idx in hidden_tabs else ""
    _chart_type.layout.display = "none" if idx in _CHART_TOGGLE_HIDDEN_TABS else ""


def _update_chart_type_options(*_):
    idx = _data_tabs.selected_index if "_data_tabs" in globals() else 0
    current = _chart_type.value
    options = _chart_options_for_tab(idx)
    valid = _option_values(options)
    _chart_type.options = options
    _chart_type.value = current if current in valid else valid[0]
    if callable(globals().get("_rebuild_chart_sub_options")):
        _rebuild_chart_sub_options()
    _sync_plot_option_visibility(idx)


for _co_idx in range(24):
    _default_row = _DEFAULT_COOCCURRENCE_ROWS[_co_idx] if _co_idx < len(_DEFAULT_COOCCURRENCE_ROWS) else {}
    _use_cb = widgets.Checkbox(
        value=bool(_default_row.get("use", _co_idx < 2)),
        description=f"Use {_co_idx + 1}",
        indent=False,
        layout=widgets.Layout(width="70px", min_width="70px"),
    )
    _default_metric = str(_default_row.get("metric") or "")
    _default_type = str(_default_row.get("type") or (_mtype(_default_metric, None) if _default_metric else ""))
    _type_dd_co = _cooccurrence_type_dropdown(_default_type)
    _metric_dd_co = _cooccurrence_metric_dropdown()
    _lo_wt = _cooccurrence_bound_text("Lo:")
    _hi_wt = _cooccurrence_bound_text("Hi:")
    _lo_wt.value = str(_default_row.get("lo", "") or "")
    _hi_wt.value = str(_default_row.get("hi", "") or "")
    _top_row = widgets.HBox(
        [_use_cb, _type_dd_co, _metric_dd_co],
        layout=widgets.Layout(width=_COOCCURRENCE_ROW_TOP_WIDTH, gap="6px", align_items="center"),
    )
    _bottom_row = widgets.HBox(
        [_lo_wt, _hi_wt],
        layout=widgets.Layout(width=_COOCCURRENCE_ROW_BOTTOM_WIDTH, gap="8px", margin="0 0 0 245px"),
    )
    _box = widgets.VBox(
        [_top_row, _bottom_row],
        layout=widgets.Layout(width=_COOCCURRENCE_PANEL_WIDTH, margin="0 0 4px 0"),
    )
    _cooccurrence_rows.append({"use": _use_cb, "type": _type_dd_co, "metric": _metric_dd_co, "lo": _lo_wt, "hi": _hi_wt, "box": _box})


for _co_index, _co_row in enumerate(_cooccurrence_rows):
    _co_row["type"].observe(_update_cooccurrence_metric_options, names="value")
    _co_row["metric"].observe(lambda change, idx=_co_index: _cooccurrence_metric_changed(change, idx), names="value")
    _co_row["use"].observe(_update_cooccurrence_info, names="value")
    _co_row["lo"].observe(_update_cooccurrence_info, names="value")
    _co_row["hi"].observe(_update_cooccurrence_info, names="value")
_cooccurrence_rows_box.children = tuple(row["box"] for row in _cooccurrence_rows)


_cooccurrence_logic.observe(_update_cooccurrence_info, names="value")
_cooccurrence_min_count.observe(_update_cooccurrence_info, names="value")
_cooccurrence_num_metrics.observe(_sync_cooccurrence_row_visibility, names="value")
_sync_cooccurrence_logic_controls()
_sync_cooccurrence_row_visibility()

In [10]:
# Plotting Helpers

def _run_shades(system: str, runs: list[str], base_color: str) -> dict[str, str]:
    base = np.array(mcolors.to_rgb(base_color))
    shades = {}
    n = max(len(runs), 1)
    for i, run in enumerate(runs):
        t = 0.5 + 0.75 * (i / max(n - 1, 1))
        shade = base * t + np.ones(3) * (1 - t) * 0.15
        shades[run] = mcolors.to_hex(np.clip(shade, 0, 1))
    return shades


def _ci95(values) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    return float(stats.t.ppf(0.975, df=len(arr) - 1) * stats.sem(arr)) if len(arr) >= 2 else 0.0


def _se_err(values) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    return float(stats.sem(arr)) if len(arr) >= 2 else 0.0


def _err_bar(values, mode: str) -> float:
    return _ci95(values) if mode == "95% CI" else _se_err(values)


def _finalize(ax: plt.Axes, title: str | None, xlabel: str | None, ylabel: str | None, xlim=None, ylim=None) -> None:
    if title:
        ax.set_title(title)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlim and xlim[0] is not None:
        ax.set_xlim(left=xlim[0])
    if xlim and xlim[1] is not None:
        ax.set_xlim(right=xlim[1])
    if ylim and ylim[0] is not None:
        ax.set_ylim(bottom=ylim[0])
    if ylim and ylim[1] is not None:
        ax.set_ylim(top=ylim[1])


def _column_name_label(metric: str, meta: dict | None = None, include_unit: bool = False) -> str:
    label = str(metric)
    if include_unit:
        unit = _MTYPE_UNIT.get(_mtype(metric, meta), "") if callable(globals().get("_mtype")) else ""
        if unit:
            label = f"{label} ({unit})"
    return label


def plot_bar(metric: str, systems: list[str], stats_data: dict, color_map: dict[str, str],
             title: str | None = None, xlabel: str | None = None, ylabel: str | None = None,
             show_dots: bool = True, error_mode: str = "SEM", xlim=None, ylim=None,
             fig_size: tuple | None = None, meta: dict | None = None,
             use_legend: bool = False) -> plt.Figure:
    if fig_size is None:
        fig_size = (max(4, len(systems)*0.75 + 0.5), 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    means, errs, colors = [], [], []
    for system in systems:
        vals = stats_data.get(system, {}).get(metric, [])
        mean, _ = mean_sem(vals)
        means.append(mean)
        errs.append(_err_bar(vals, error_mode))
        colors.append(color_map.get(system, "#888"))
    x = np.arange(len(systems))
    if use_legend:
        for i, (system, mean, err, color) in enumerate(zip(systems, means, errs, colors)):
            ax.bar(x[i], mean, yerr=err, color=color, capsize=5, width=0.6, edgecolor="white", linewidth=0.5, label=system)
    else:
        ax.bar(x, means, yerr=errs, color=colors, capsize=5, width=0.6, edgecolor="white", linewidth=0.5)
    if show_dots:
        for i, system in enumerate(systems):
            for value in stats_data.get(system, {}).get(metric, []):
                ax.plot(i, value, "o", color="#8a8a8a", ms=5, alpha=0.85, zorder=3)
    ax.set_xticks(x)
    if use_legend:
        ax.set_xticklabels([""] * len(systems))
        ax.tick_params(axis="x", length=0)
    else:
        ax.set_xticklabels(systems, rotation=35, ha="right")
    ax.set_ylabel(ylabel or _column_name_label(metric, meta))
    ax.set_title(title or str(metric))
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig




def _smooth_series_values(values: np.ndarray, window: int) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if window < 2 or len(arr) < 3:
        return arr.copy()
    if window % 2 == 0:
        window += 1
    window = min(window, len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    if window < 3:
        return arr.copy()
    pad = window // 2
    padded = np.pad(arr, pad, mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(padded, kernel, mode="valid")


def plot_timeseries(metric: str, systems: list[str], color_map: dict[str, str], runs: list[str] | None = None,
                    individual: bool = False, root: Path = RESULTS_ROOT, error_mode: str = "SEM",
                    smooth_window: int = 1,
                    title: str | None = None, xlabel: str | None = None, ylabel: str | None = None,
                    xlim=None, ylim=None, fig_size: tuple | None = None, meta: dict | None = None) -> plt.Figure:
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    for system in systems:
        arrays: list[tuple[str, np.ndarray]] = []
        for run in (runs or discover_runs(system, root)):
            arr = load_run_raw(system, run, metric, root)
            if arr is not None:
                if _is_hbond_metric(metric):
                    arr = arr * 100.0
                arrays.append((run, arr))
        if not arrays:
            continue
        color = color_map.get(system, "#888")
        if individual:
            shades = _run_shades(system, [r for r, _ in arrays], color)
            for run, arr in arrays:
                y = _smooth_series_values(arr, smooth_window)
                ax.plot(np.arange(len(y)), y, lw=1, color=shades[run], label=f"{system} / {run}", alpha=0.85)
        else:
            n = min(len(a) for _, a in arrays)
            mat = np.stack([a[:n] for _, a in arrays])
            mean_arr = np.nanmean(mat, axis=0)
            err_arr = (np.array([_ci95(mat[:, j]) for j in range(n)]) if error_mode == "95% CI" and mat.shape[0] > 1
                       else (np.nanstd(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0]) if mat.shape[0] > 1 else np.zeros(n)))
            mean_arr = _smooth_series_values(mean_arr, smooth_window)
            err_arr = _smooth_series_values(err_arr, smooth_window)
            frames = np.arange(len(mean_arr))
            ax.plot(frames, mean_arr, lw=1, color=color, label=system)
            ax.fill_between(frames, mean_arr - err_arr, mean_arr + err_arr, color=color, alpha=0.20)
    ax.set_xlabel(xlabel or "Frame")
    ax.set_ylabel(ylabel or _column_name_label(metric, meta))
    ax.set_title(title or f"{metric} - Time Series")
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def plot_kde(metric: str, systems: list[str], color_map: dict[str, str], runs: list[str] | None = None,
             individual: bool = False, root: Path = RESULTS_ROOT, title: str | None = None,
             xlabel: str | None = None, ylabel: str | None = None, xlim=None, ylim=None,
             fig_size: tuple | None = None, meta: dict | None = None) -> plt.Figure:
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    for system in systems:
        arrays: list[tuple[str, np.ndarray]] = []
        for run in (runs or discover_runs(system, root)):
            arr = load_run_raw(system, run, metric, root)
            if arr is not None:
                arrays.append((run, arr[np.isfinite(arr)]))
        if not arrays:
            continue
        color = color_map.get(system, "#888")
        if individual:
            shades = _run_shades(system, [r for r, _ in arrays], color)
            for run, arr in arrays:
                if len(arr) >= 5:
                    sns.kdeplot(arr, ax=ax, color=shades[run], label=f"{system} / {run}", lw=1, fill=True, alpha=0.08)
        else:
            combined = np.concatenate([a for _, a in arrays])
            combined = combined[np.isfinite(combined)]
            if len(combined) >= 5:
                sns.kdeplot(combined, ax=ax, color=color, label=system, lw=1, fill=True, alpha=0.15)
    ax.set_xlabel(xlabel or _column_name_label(metric, meta))
    ax.set_ylabel(ylabel or "Density")
    ax.set_title(title or f"{metric} - Distribution")
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def plot_rdfs(records: list[dict], systems: list[str], color_map: dict[str, str], runs: list[str] | None = None,
              individual: bool = False, root: Path = RESULTS_ROOT, title: str | None = None,
              xlabel: str | None = None, ylabel: str | None = None, xlim=None, ylim=None,
              fig_size: tuple | None = None, smooth_sigma: float = 1.0) -> plt.Figure:

    def _smooth(arr: np.ndarray) -> np.ndarray:
        arr = arr.astype(float)
        return arr if smooth_sigma <= 0 else gaussian_filter1d(arr, sigma=smooth_sigma)

    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    if not records:
        ax.text(0.5, 0.5, "No RDF records selected", ha="center", va="center", transform=ax.transAxes)
        return fig
    for system in systems:
        color = color_map.get(system, "#888")
        shades = _run_shades(system, [r for r in (runs or discover_runs(system, root))], color)
        for rec_i, rec in enumerate(records):
            key = rec["key"]
            label_base = _rdf_record_display(rec)
            dfs: list[tuple[str, pd.DataFrame]] = []
            for run in (runs or discover_runs(system, root)):
                df = load_rdf_series(system, run, key, root)
                if df is not None and {"r_ang", "g_rdf"}.issubset(df.columns):
                    dfs.append((run, df))
            if not dfs:
                continue
            if individual:
                for run, df in dfs:
                    ax.plot(df["r_ang"], _smooth(df["g_rdf"].to_numpy()), lw=1,
                            color=shades.get(run, color), alpha=0.75,
                            label=f"{system} / {run} - {label_base}")
            else:
                r = dfs[0][1]["r_ang"].to_numpy()
                g_stack = np.stack([df["g_rdf"].to_numpy() for _, df in dfs])
                g_mean = _smooth(np.nanmean(g_stack, axis=0))
                g_err = (_smooth(np.nanstd(g_stack, axis=0, ddof=1) / np.sqrt(g_stack.shape[0]))
                         if g_stack.shape[0] > 1 else np.zeros_like(g_mean))
                ls = ["-", "--", ":", "-."][rec_i % 4]
                ax.plot(r, g_mean, lw=1, ls=ls, color=color, label=f"{system} - {label_base}")
                ax.fill_between(r, g_mean - g_err, g_mean + g_err, color=color, alpha=0.10)
    ax.set_xlabel(xlabel or "r (Å)")
    ax.set_ylabel(ylabel or "g(r)")
    ax.set_title(title or "RDF")
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


if not hasattr(_MplAxes, "_cmp_original_legend"):
    _MplAxes._cmp_original_legend = _MplAxes.legend


def _legend_below(ax: plt.Axes, *args, **kwargs):
    handles, labels = ax.get_legend_handles_labels()
    if not args and not handles:
        return None
    kwargs.pop("loc", None)
    kwargs.setdefault("frameon", False)
    kwargs.setdefault("bbox_to_anchor", (0.5, -0.18))
    kwargs.setdefault("borderaxespad", 0.0)
    _fig = ax.get_figure()
    _fig_w = _fig.get_figwidth() if _fig is not None else 8.0
    _n_labels = max(1, len(labels) if labels else 1)
    kwargs.setdefault("ncol", min(max(1, int(_fig_w / 3.0)), _n_labels))
    legend = _MplAxes._cmp_original_legend(ax, *args, loc="upper center", **kwargs)
    legend.set_in_layout(True)
    return legend


def _cmp_legend_below(self, *args, **kwargs):
    return _legend_below(self, *args, **kwargs)


_MplAxes.legend = _cmp_legend_below


In [11]:
# Speed Cache Wrappers

_RUNS_CACHE = globals().get("_RUNS_CACHE", {})
_RUN_MEANS_CACHE = globals().get("_RUN_MEANS_CACHE", {})
_STATS_CACHE = globals().get("_STATS_CACHE", {})
_INDIVIDUAL_CACHE = globals().get("_INDIVIDUAL_CACHE", {})
_METRIC_COLUMNS_CACHE = globals().get("_METRIC_COLUMNS_CACHE", {})
_RDF_CATALOG_CACHE = globals().get("_RDF_CATALOG_CACHE", {})
_RMS_CATALOG_CACHE = globals().get("_RMS_CATALOG_CACHE", {})
_BASE_DISCOVER_RUNS = globals().setdefault("_BASE_DISCOVER_RUNS", discover_runs)
_BASE_LOAD_RUN_MEANS = globals().setdefault("_BASE_LOAD_RUN_MEANS", load_run_means)
_BASE_COLLECT_STATS = globals().setdefault("_BASE_COLLECT_STATS", collect_stats)
_BASE_COLLECT_INDIVIDUAL = globals().setdefault("_BASE_COLLECT_INDIVIDUAL", collect_individual)


def _run_cache_key(system: str, run: str, root: Path = RESULTS_ROOT) -> tuple[str, str, str]:
    return (str(root), system, run)


def discover_runs(system: str, root: Path = RESULTS_ROOT) -> list[str]:
    key = (str(root), system)
    if key not in _RUNS_CACHE:
        _RUNS_CACHE[key] = tuple(_BASE_DISCOVER_RUNS(system, root))
    return list(_RUNS_CACHE[key])


def load_run_means(system: str, run: str, root: Path = RESULTS_ROOT) -> dict[str, float] | None:
    key = _run_cache_key(system, run, root)
    if key not in _RUN_MEANS_CACHE:
        _RUN_MEANS_CACHE[key] = _BASE_LOAD_RUN_MEANS(system, run, root)
    return _RUN_MEANS_CACHE[key]


def _stats_cache_key(systems: list[str], runs: list[str] | None = None, root: Path = RESULTS_ROOT) -> tuple:
    return (str(root), tuple(systems), None if runs is None else tuple(runs))


def collect_stats(systems: list[str], runs: list[str] | None = None, root: Path = RESULTS_ROOT) -> dict[str, dict[str, list[float]]]:
    key = _stats_cache_key(systems, runs, root)
    if key not in _STATS_CACHE:
        _STATS_CACHE[key] = _BASE_COLLECT_STATS(systems, runs, root)
    return _STATS_CACHE[key]


def collect_individual(systems: list[str], runs: list[str] | None = None, root: Path = RESULTS_ROOT) -> list[tuple[str, str, dict[str, float]]]:
    key = _stats_cache_key(systems, runs, root)
    if key not in _INDIVIDUAL_CACHE:
        _INDIVIDUAL_CACHE[key] = _BASE_COLLECT_INDIVIDUAL(systems, runs, root)
    return _INDIVIDUAL_CACHE[key]


def _all_current_metric_columns(systems: list[str]) -> list[str]:
    columns: set[str] = set()
    for system in systems:
        for run in discover_runs(system, RESULTS_ROOT):
            key = _run_cache_key(system, run, RESULTS_ROOT)
            if key in _METRIC_COLUMNS_CACHE:
                cols = _METRIC_COLUMNS_CACHE[key]
            df = _load_df(system, run, RESULTS_ROOT)
            if df is not None:
                cols = tuple(c for c in df.columns if not _is_hidden_metric(c))
            else:
                csv_path = RESULTS_ROOT / system / run / "Active_Metrics" / "metrics.csv"
                cols = tuple(c for c in pd.read_csv(csv_path, nrows=0).columns if not _is_hidden_metric(c)) if csv_path.exists() else tuple()
            _METRIC_COLUMNS_CACHE[key] = cols
            columns.update(cols)
    return sorted(list(columns))


In [12]:
# Patch: keep individual-run bar plots compatible with legend controls.
def plot_bar_individual(metric: str, individual_data: list[tuple[str, str, dict]], color_map: dict[str, str],
                        title: str | None = None, xlabel: str | None = None, ylabel: str | None = None,
                        xlim=None, ylim=None, fig_size: tuple | None = None, meta: dict | None = None,
                        use_legend: bool = False) -> plt.Figure:
    if fig_size is None:
        fig_size = (max(4, len(individual_data) * 0.75 + 0.5), 2.5)

    fig, ax = plt.subplots(figsize=fig_size)
    x = np.arange(len(individual_data))
    seen_systems: set[str] = set()

    runs_by_system: dict[str, list[str]] = {}
    for system, run, _ in individual_data:
        runs_by_system.setdefault(system, []).append(run)

    run_shades: dict[tuple[str, str], str] = {}
    for system, runs in runs_by_system.items():
        shades = _run_shades(system, runs, color_map.get(system, "#888"))
        for run in runs:
            run_shades[(system, run)] = shades.get(run, color_map.get(system, "#888"))

    for i, (system, run, data) in enumerate(individual_data):
        value = data.get(metric, np.nan)
        color = run_shades.get((system, run), color_map.get(system, "#888"))
        label = system if use_legend and system not in seen_systems else None
        seen_systems.add(system)
        ax.bar(x[i], value, color=color, width=0.6, edgecolor="white", linewidth=0.5, label=label)

    ax.set_xticks(x)
    if use_legend:
        ax.set_xticklabels([run for _, run, _ in individual_data], rotation=40, ha="right", fontsize=8)
        ax.legend(loc="best", ncols=2, frameon=False)
    else:
        ax.set_xticklabels([f"{system}\n{run}" for system, run, _ in individual_data], rotation=40, ha="right", fontsize=8)

    ax.set_ylabel(ylabel or _column_name_label(metric, meta))
    ax.set_title(title or f"{metric} (individual runs)")
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig

In [13]:
# Contact Map and Threshold Occupancy

_THRESHOLD_OCC_CACHE: dict[tuple, float | None] = globals().get("_THRESHOLD_OCC_CACHE", {})


def _is_ptk_peptide_contact_map_metric(metric: str, meta: dict | None = None) -> bool:
    row = (meta or {}).get(metric, {}) or {}
    category = str(row.get("category") or "").strip().lower()
    section = str(row.get("plot_section") or "").strip().lower()
    base_name = str(row.get("base_metric_name") or "").strip().lower()
    output_kind = str(row.get("output_kind") or "").strip().lower()
    operation = str(row.get("operation") or "").strip().lower()
    if _CONTACT_MAP_CATEGORY in {category, section}:
        return True
    if category == "ptk_peptide" and output_kind in {"distance", "hbond_occupancy"}:
        return True
    if base_name == "ptk_peptide" and operation in {"distance", "hbond"}:
        return True
    return str(metric).startswith("ptk_peptide") and (metric.endswith("_dist") or "_hbond" in metric)


def _contact_map_distance_metric(metric: str, meta: dict | None = None) -> str | None:
    if not _is_ptk_peptide_contact_map_metric(metric, meta):
        return None
    metric = str(metric)
    if metric.endswith("_dist"):
        return metric
    for hb_suf in ("_hbond_sc", "_hbond_bb", "_hbond_mix", "_hbond"):
        if metric.endswith(hb_suf):
            return metric[:-len(hb_suf)] + "_dist"
    return None


def _contact_run_fraction_below(system: str, run: str, distance_metric: str, cutoff: float, root: Path = RESULTS_ROOT) -> float | None:
    key = (*_run_cache_key(system, run, root), distance_metric, round(float(cutoff), 4)) if callable(globals().get("_run_cache_key")) else (str(root), system, run, distance_metric, round(float(cutoff), 4))
    if key not in _CONTACT_OCC_CACHE:
        arr = load_run_raw(system, run, distance_metric, root) if callable(globals().get("load_run_raw")) else None
        if arr is None:
            _CONTACT_OCC_CACHE[key] = None
        else:
            arr = np.asarray(arr, dtype=float)
            arr = arr[np.isfinite(arr)]
            _CONTACT_OCC_CACHE[key] = float(np.mean(arr < float(cutoff))) if arr.size else None
    return _CONTACT_OCC_CACHE[key]


def _install_contact_map_controls() -> None:
    return


def _contact_filter_changed(*_) -> None:
    _CONTACT_OCC_CACHE.clear()
    if callable(globals().get("_schedule_metric_update")):
        _schedule_metric_update()

def _on_systems_change(*_) -> None:
    global _system_update_timer
    if _system_update_timer is not None:
        _system_update_timer.cancel()
    _system_update_timer = threading.Timer(0.25, _do_system_dependent_update)
    _system_update_timer.start()


def _threshold_bounds_for_metric(metric: str, meta: dict | None = None) -> tuple[float | None, float | None]:
    return float(_thresh_lo_wt.value), float(_thresh_hi_wt.value)


def _threshold_pct_array(arr: np.ndarray | None, lo: float | None, hi: float | None) -> float | None:
    if arr is None:
        return None
    values = np.asarray(arr, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0 or (lo is None and hi is None):
        return None
    mask = np.ones(values.shape, dtype=bool)
    if lo is not None:
        mask &= values >= float(lo)
    if hi is not None:
        mask &= values <= float(hi)
    return float(mask.mean() * 100.0)


def _threshold_pct_for_run(system: str, run: str, metric: str, lo: float | None, hi: float | None, root: Path = RESULTS_ROOT) -> float | None:
    key = (*_run_cache_key(system, run, root), metric, lo, hi) if callable(globals().get("_run_cache_key")) else (str(root), system, run, metric, lo, hi)
    if key not in _THRESHOLD_OCC_CACHE:
        _THRESHOLD_OCC_CACHE[key] = _threshold_pct_array(load_run_raw(system, run, metric, root), lo, hi)
    return _THRESHOLD_OCC_CACHE[key]


def _threshold_pct_for_rdf_charge_group(system: str, run: str, metrics: list[str], lo: float | None, hi: float | None, root: Path = RESULTS_ROOT) -> float | None:
    key = (*_run_cache_key(system, run, root), tuple(metrics), lo, hi) if callable(globals().get("_run_cache_key")) else (str(root), system, run, tuple(metrics), lo, hi)
    if key not in _THRESHOLD_OCC_CACHE:
        combined = _load_rdf_charge_group_raw(system, run, metrics, "distance", root) if callable(globals().get("_load_rdf_charge_group_raw")) else None
        _THRESHOLD_OCC_CACHE[key] = _threshold_pct_array(combined, lo, hi)
    return _THRESHOLD_OCC_CACHE[key]


def _collect_threshold_pct_stats(systems: list[str], metric: str, lo: float | None, hi: float | None, root: Path = RESULTS_ROOT) -> dict[str, dict[str, list[float]]]:
    result: dict[str, dict[str, list[float]]] = {}
    meta = _get_shared_meta(systems)
    grouped = callable(globals().get("_is_rdf_charge_group_value")) and _is_rdf_charge_group_value(metric)
    members = _rdf_charge_group_members(metric, systems, meta) if grouped else []
    for system in systems:
        values: list[float] = []
        for run in discover_runs(system, root):
            pct = (_threshold_pct_for_rdf_charge_group(system, run, members, lo, hi, root)
                   if grouped else _threshold_pct_for_run(system, run, metric, lo, hi, root))
            if pct is not None:
                values.append(pct)
        if values:
            result[system] = {"threshold_pct": values}
    return result


def _collect_threshold_pct_individual(systems: list[str], metric: str, lo: float | None, hi: float | None, root: Path = RESULTS_ROOT) -> list[tuple[str, str, dict[str, float]]]:
    rows: list[tuple[str, str, dict[str, float]]] = []
    meta = _get_shared_meta(systems)
    grouped = callable(globals().get("_is_rdf_charge_group_value")) and _is_rdf_charge_group_value(metric)
    members = _rdf_charge_group_members(metric, systems, meta) if grouped else []
    for system in systems:
        for run in discover_runs(system, root):
            pct = (_threshold_pct_for_rdf_charge_group(system, run, members, lo, hi, root)
                   if grouped else _threshold_pct_for_run(system, run, metric, lo, hi, root))
            if pct is not None:
                rows.append((system, run, {"threshold_pct": pct}))
    return rows


for _cb in _sys_checks.values():
    _cb.observe(_on_systems_change, names="value")


for _widget in (_contact_filter_cb, _contact_cutoff_wt, _contact_min_pct_wt):
    _widget.observe(_contact_filter_changed, names="value")


_install_contact_map_controls()

In [14]:
# Contact Filter: Distance vs H-Bond Occupancy

_CONTACT_FILTER_DEFAULT_MSG = "Passes when at least one selected run is below cutoff for the requested trajectory fraction."
_CONTACT_FILTER_HBOND_MSG = "Passes when at least one selected run has H-bond occupancy at or above the requested trajectory fraction."


def _contact_filter_is_hbond_mode() -> bool:
    mtype_value = getattr(_mtype_filter_dd, "value", "") if "_mtype_filter_dd" in globals() else ""
    if mtype_value == "H-Bond (%)":
        return True
    metric = getattr(_metric_dd, "value", None) if "_metric_dd" in globals() else None
    if not metric:
        return False
    systems = _get_selected_systems() if callable(globals().get("_get_selected_systems")) else []
    meta = _get_shared_meta(systems) if systems and callable(globals().get("_get_shared_meta")) else {}
    return _mtype(metric, meta) == "hbond"




def _contact_filter_changed(*_) -> None:
    hbond_mode = _contact_filter_is_hbond_mode()
    _contact_filter_cb.description = "Filter options by H-bond occupancy" if hbond_mode else "Filter options by distance occupancy"
    _contact_cutoff_wt.layout.display = "none" if hbond_mode else ""
    _contact_min_pct_wt.description = "Min traj %:"
    msg = _CONTACT_FILTER_HBOND_MSG if hbond_mode else _CONTACT_FILTER_DEFAULT_MSG
    _contact_filter_msg.value = f"<span style='font-size:11px;color:#aaa'>{msg}</span>"
    _CONTACT_OCC_CACHE.clear()
    if callable(globals().get("_schedule_metric_update")):
        _schedule_metric_update()


def _contact_run_hbond_fraction(system: str, run: str, hbond_metric: str, root: Path = RESULTS_ROOT) -> float | None:
    key = (*_run_cache_key(system, run, root), hbond_metric, "hbond_fraction") if callable(globals().get("_run_cache_key")) else (str(root), system, run, hbond_metric, "hbond_fraction")
    if key not in _CONTACT_OCC_CACHE:
        arr = load_run_raw(system, run, hbond_metric, root) if callable(globals().get("load_run_raw")) else None
        if arr is None:
            _CONTACT_OCC_CACHE[key] = None
        else:
            values = np.asarray(arr, dtype=float)
            values = values[np.isfinite(values)]
            if values.size == 0:
                _CONTACT_OCC_CACHE[key] = None
            else:
                _CONTACT_OCC_CACHE[key] = float(np.mean(values > 0.0))
    return _CONTACT_OCC_CACHE[key]


def _metric_passes_contact_occupancy(metric: str, systems: list[str], meta: dict | None = None) -> bool:
    if not _contact_filter_cb.value:
        return True
    metric_type = _mtype(metric, meta)
    hbond_mode = _contact_filter_is_hbond_mode()
    min_fraction = float(_contact_min_pct_wt.value) / 100.0
    if hbond_mode:
        if metric_type != "hbond":
            return False
        for system in systems:
            for run in discover_runs(system, RESULTS_ROOT):
                fraction = _contact_run_hbond_fraction(system, run, metric, RESULTS_ROOT)
                if fraction is not None and fraction >= min_fraction:
                    return True
        return False
    distance_metric = _contact_map_distance_metric(metric, meta) if _is_ptk_peptide_contact_map_metric(metric, meta) else (str(metric) if str(metric).endswith("_dist") else None)
    if not distance_metric:
        return False
    cutoff = float(_contact_cutoff_wt.value)
    for system in systems:
        for run in discover_runs(system, RESULTS_ROOT):
            fraction = _contact_run_fraction_below(system, run, distance_metric, cutoff, RESULTS_ROOT)
            if fraction is not None and fraction >= min_fraction:
                return True
    return False


for _widget in (_contact_filter_cb, _contact_cutoff_wt, _contact_min_pct_wt, _mtype_filter_dd, _metric_dd):
    try:
        _widget.observe(_contact_filter_changed, names="value")
    except Exception:
        pass


_contact_filter_changed()


In [15]:
# RMSD and RMSF

_RMS_CACHE: dict[tuple, pd.DataFrame | None] = {}
_RMS_FILE_PATTERNS = ("*rmsd*.xvg", "*rmsf*.xvg", "*rmsd*.csv", "*rmsf*.csv")


def _rms_files_in_dir(rms_dir: Path) -> list[Path]:
    files: list[Path] = []
    for pattern in _RMS_FILE_PATTERNS:
        files.extend(path for path in rms_dir.glob(pattern) if "rms" in path.stem.lower())
    return sorted(files)


def discover_rms_systems(root: Path = RESULTS_ROOT) -> list[str]:
    if not root.is_dir():
        return []
    systems: list[str] = []
    for system_dir in root.iterdir():
        if not system_dir.is_dir() or system_dir.name.startswith(".") or system_dir.name == "Comparison_Plots":
            continue
        if any(_rms_files_in_dir(run_dir / "RMSD") for run_dir in system_dir.iterdir() if run_dir.is_dir()):
            systems.append(system_dir.name)
    return sorted(systems)


def discover_rms_runs(system: str, root: Path = RESULTS_ROOT) -> list[str]:
    system_dir = root / system
    if not system_dir.is_dir():
        return []
    runs: list[str] = []
    for run_dir in sorted(system_dir.iterdir()):
        if run_dir.is_dir() and _rms_files_in_dir(run_dir / "RMSD"):
            runs.append(run_dir.name)
    return runs


def _install_rms_systems() -> None:
    if "_ALL_SYSTEMS" not in globals() or "_rebuild_sys_checkboxes" not in globals():
        return
    rms_systems = discover_rms_systems(RESULTS_ROOT)
    if not rms_systems:
        return
    current = set(_ALL_SYSTEMS)
    added = [system for system in rms_systems if system not in current]
    if added:
        _ALL_SYSTEMS[:] = sorted(set(_ALL_SYSTEMS).union(added))
        _rebuild_sys_checkboxes()
    if callable(globals().get("_on_systems_change")):
        for cb in _sys_checks.values():
            cb.observe(_on_systems_change, names="value")


def _parse_rms_xvg(path: Path, kind: str) -> pd.DataFrame | None:
    xs, ys = [], []
    try:
        with open(path) as fh:
            for line in fh:
                line = line.strip()
                if not line or line.startswith("#") or line.startswith("@"):
                    continue
                parts = line.split()
                if len(parts) < 2:
                    continue
                xs.append(float(parts[0]))
                ys.append(float(parts[1]))
    except Exception:
        return None
    if not xs:
        return None
    x = np.asarray(xs, dtype=float)
    y = np.asarray(ys, dtype=float) * 10.0
    if kind == "rmsd":
        x = x / 1000.0
        x_col = "time_ns"
    else:
        x_col = "atom"
    return pd.DataFrame({x_col: x, "value_a": y})


def _parse_rms_csv(path: Path, kind: str) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    numeric = df.select_dtypes(include="number").columns.tolist()
    if not numeric:
        return None
    cols = {c.lower(): c for c in df.columns}
    if kind == "rmsd":
        x_col = next((cols[k] for k in ("time_ns", "time", "t") if k in cols), numeric[0])
        out_x_col = "time_ns"
    else:
        x_col = next((cols[k] for k in ("atom", "index", "res_pos", "residue") if k in cols), numeric[0])
        out_x_col = "atom"
    y_col = next((cols[k] for k in ("value_a", "rmsf", "rmsd", "rms", "r") if k in cols), None)
    if y_col is None:
        y_col = numeric[1] if len(numeric) > 1 else numeric[0]
    x = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)
    if kind == "rmsd" and np.nanmax(x) > 1000:
        x = x / 1000.0
    if np.nanmax(y) < 2.0:
        y = y * 10.0
    return pd.DataFrame({out_x_col: x, "value_a": y})


def _parse_pdb_serials(path: Path) -> dict[int, tuple[int, str, str]]:
    serial_map: dict[int, tuple[int, str, str]] = {}
    if not path.exists():
        return serial_map
    with open(path) as fh:
        for line in fh:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            try:
                serial = int(line[6:11])
                atom_name = line[12:16].strip()
                resname = line[17:20].strip()
                resid = int(line[22:26])
            except ValueError:
                continue
            serial_map[serial] = (resid, resname, atom_name)
    return serial_map


def _rms_display_name(filename: str) -> tuple[str, str]:
    stem = Path(filename).stem
    clean = stem
    clean = re.sub(r"(?i)^rms[fd]?_?", "", clean)
    clean = re.sub(r"(?i)_?rms[fd]$", "", clean)
    clean = clean.strip("_")
    if not clean:
        return "Backbone", ""
    parts = [p for p in clean.split("_") if p]
    if len(parts) >= 2:
        group = parts[0].upper() if parts[0].lower() in {"ptk", "atp"} else parts[0].capitalize()
        target = " ".join(parts[1:]).replace("-", " ")
    else:
        group = parts[0].upper() if parts[0].lower() in {"ptk", "atp"} else parts[0].capitalize()
        target = ""
    target = target.replace("ptr", "pY").replace("tyr", "Y")
    return group, target


def _annotate_rmsf_atoms(df: pd.DataFrame, pdb_path: Path, filename: str, system: str) -> pd.DataFrame:
    def _sequential_out(src: pd.DataFrame, label: str = "Atom serial index") -> pd.DataFrame:
        out = src.copy()
        n = len(out)
        out["resnum"] = np.arange(1, n + 1)
        out["resname"] = ["RES"] * n
        out["atom_name"] = ["A"] * n
        out["res_pos"] = out["resnum"].astype(int)
        out["x"] = out["res_pos"].astype(float)
        out["x_label"] = out["resnum"].astype(str)
        out.attrs["axis_label"] = label
        out.attrs["axis_mode"] = "sequential"
        out.attrs["axis_ref"] = ""
        return out

    out = df.copy()
    atom_ids = pd.to_numeric(out["atom"], errors="coerce").to_numpy(dtype=float)
    if atom_ids.size == 0 or np.isnan(atom_ids).all():
        return _sequential_out(out)
    serial_map = _parse_pdb_serials(pdb_path)
    if not serial_map:
        return _sequential_out(out)
    mapped = [serial_map.get(int(atom), (None, "UNK", "")) if np.isfinite(atom) else (None, "UNK", "") for atom in atom_ids]
    out["resnum"] = [m[0] for m in mapped]
    out["resname"] = [m[1] for m in mapped]
    out["atom_name"] = [m[2] for m in mapped]
    if out["resnum"].isna().any():
        missing = out["resnum"].isna()
        out.loc[missing, "resnum"] = np.arange(1, missing.sum() + 1)
    out["resnum"] = pd.to_numeric(out["resnum"], errors="coerce").fillna(0).astype(int)
    unique_resnums = list(dict.fromkeys(out["resnum"].tolist()))
    seq_map = {resnum: idx + 1 for idx, resnum in enumerate(unique_resnums)}
    base_name = filename
    if base_name.startswith("rmsf_"):
        base_name = base_name[len("rmsf_"):]
    if base_name.endswith(".xvg"):
        base_name = base_name[:-4]
    if base_name.endswith(".csv"):
        base_name = base_name[:-4]
    if base_name.endswith("_rmsf"):
        base_name = base_name[:-5]
    base_name = base_name.strip("_")
    group_key = base_name.split("_", 1)[0].lower() if "_" in base_name else base_name.lower()
    residue_groups = _CONFIG.get("residue_groups") or {}
    motif_groups = _CONFIG.get("motif_groups") or {}
    residue_key = {str(k).lower(): k for k in residue_groups}.get(group_key)
    motif_key = {str(k).lower(): k for k in motif_groups}.get(group_key)
    axis_mode = "sequential"
    axis_ref = None
    ref_resid = None
    ref_index = None
    if residue_key is not None:
        group_def = residue_groups[residue_key] or {}
        axis_mode = group_def.get("numbering") or "sequential"
        axis_ref = group_def.get("rel_ref")
    elif motif_key is not None:
        for member in (motif_groups.get(motif_key) or []):
            group_def = residue_groups.get(member) or {}
            if group_def.get("numbering") == "relative":
                axis_mode = "relative"
                axis_ref = group_def.get("rel_ref")
                break
    if axis_mode == "relative" and axis_ref:
        residues = _CONFIG.get("residues") or {}
        entry = residues.get(str(axis_ref)) or {}
        system_cfg = (_CONFIG.get("systems") or {}).get(system, {})
        lookup_keys = (
            system_cfg.get("kinase_key"),
            system_cfg.get("family"),
            system.split("_", 1)[0],
        )
        ref_entry = next((entry[key] for key in lookup_keys if key in entry), {})

        try:
            ref_resid = int(ref_entry.get("resid")) if ref_entry.get("resid") is not None else None
        except (TypeError, ValueError):
            ref_resid = None
        try:
            ref_index = int(ref_entry.get("idx_source_resid")) if ref_entry.get("idx_source_resid") is not None else None
        except (TypeError, ValueError):
            ref_index = None
    if axis_mode == "relative":
        if ref_resid in unique_resnums:
            residue_pos = {r: r - int(ref_resid) for r in unique_resnums}
            axis_label = f"Residue position relative to {axis_ref} ({ref_resid})"
        elif ref_index is not None and 1 <= int(ref_index) <= len(unique_resnums):
            ref_order = int(ref_index)
            residue_pos = {r: seq_map[r] - ref_order for r in unique_resnums}
            axis_label = f"Residue position relative to {axis_ref}"
        else:
            axis_mode = "sequential"
            residue_pos = seq_map
            axis_label = "Sequential residue position"
    else:
        residue_pos = seq_map
        axis_label = "Sequential residue position"
    out["res_pos"] = out["resnum"].map(residue_pos).astype(int)
    atom_x = np.empty(len(out), dtype=float)
    for resnum in unique_resnums:
        idxs = out.index[out["resnum"] == resnum].to_list()
        rp = residue_pos[resnum]
        if len(idxs) == 1:
            atom_x[idxs[0]] = rp
        else:
            for j, idx in enumerate(idxs):
                atom_x[idx] = rp - 0.5 + j / (len(idxs) - 1)
    out["x"] = atom_x
    out["x_label"] = out["resname"] + ":" + out["atom_name"]
    out.attrs["axis_label"] = axis_label
    out.attrs["axis_mode"] = axis_mode
    out.attrs["axis_ref"] = axis_ref or ""
    return out


def discover_rms_records(systems: list[str], root: Path = RESULTS_ROOT) -> list[dict]:
    seen: set[tuple[str, str]] = set()
    records: list[dict] = []
    for system in systems:
        for run in discover_rms_runs(system, root):
            rms_dir = root / system / run / "RMSD"
            for path in _rms_files_in_dir(rms_dir):
                if "rmsf" in path.name.lower():
                    kind = "rmsf"
                elif "rmsd" in path.name.lower():
                    kind = "rmsd"
                else:
                    print(f"Warning: Unrecognized RMS file kind for {path.name} in {system}/{run}")
                    continue
                key = (kind, path.name)
                if key in seen:
                    continue
                seen.add(key)
                group_label, target_label = _rms_display_name(path.name)
                label = f"{group_label} to {target_label}" if target_label else group_label
                records.append({
                    "key": f"{kind}:{path.name}",
                    "kind": kind,
                    "filename": path.name,
                    "group_label": group_label,
                    "target_label": target_label,
                    "label": label,
                })
    records.sort(key=lambda r: (r["kind"], r["label"]))
    return records


def _load_rms_run(system: str, run: str, filename: str, kind: str, root: Path = RESULTS_ROOT) -> pd.DataFrame | None:
    key = (str(root), system, run, filename, kind)
    if key in _RMS_CACHE:
        return _RMS_CACHE[key]
    path = root / system / run / "RMSD" / filename
    if not path.exists():
        _RMS_CACHE[key] = None
        return None
    df = _parse_rms_xvg(path, kind) if path.suffix.lower() == ".xvg" else _parse_rms_csv(path, kind)
    if df is not None and kind == "rmsf":
        pdb_path = root / system / run / "Trajectories" / "md_ref.pdb"
        df = _annotate_rmsf_atoms(df, pdb_path, filename, system)
    _RMS_CACHE[key] = df
    return df


def _smooth_rms_values(values: np.ndarray, window: int) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if window < 2 or len(arr) < 3:
        return arr.copy()
    if window % 2 == 0:
        window += 1
    window = min(window, len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    if window < 3:
        return arr.copy()
    pad = window // 2
    padded = np.pad(arr, pad, mode="edge")
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(padded, kernel, mode="valid")


def plot_rms_profile(record: dict, systems: list[str], color_map: dict[str, str], individual: bool = False,
                     root: Path = RESULTS_ROOT, smooth_window: int = 1, title: str | None = None,
                     xlabel: str | None = None, ylabel: str | None = None, xlim=None, ylim=None,
                     fig_size: tuple | None = None) -> plt.Figure:
    kind = record["kind"]
    filename = record["filename"]
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    axis_label = "Atom serial index"
    for system in systems:
        runs = discover_rms_runs(system, root)
        color = color_map.get(system, "#888")
        shades = _run_shades(system, runs, color)
        dfs: list[tuple[str, pd.DataFrame]] = []
        for run in runs:
            df = _load_rms_run(system, run, filename, kind, root)
            if df is not None and not df.empty:
                dfs.append((run, df))
        if not dfs:
            continue
        if kind == "rmsf":
            axis_label = "Atom serial index"
        if individual:
            for run, df in dfs:
                if kind == "rmsd":
                    x = df["time_ns"].to_numpy(dtype=float)
                else:
                    x = df["atom"].to_numpy(dtype=float)
                y = _smooth_rms_values(df["value_a"].to_numpy(dtype=float), smooth_window)
                ax.plot(x, y, lw=1, color=shades.get(run, color), alpha=0.85, label=f"{system} / {run}")
        else:
            min_len = min(len(df) for _, df in dfs)
            if min_len == 0:
                continue
            if kind == "rmsd":
                x = dfs[0][1]["time_ns"].to_numpy(dtype=float)[:min_len]
            else:
                x = dfs[0][1]["atom"].to_numpy(dtype=float)[:min_len]
            mat = np.stack([df["value_a"].to_numpy(dtype=float)[:min_len] for _, df in dfs])
            mean_arr = _smooth_rms_values(np.nanmean(mat, axis=0), smooth_window)
            err_arr = np.nanstd(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0]) if mat.shape[0] > 1 else np.zeros(min_len)
            err_arr = _smooth_rms_values(err_arr, smooth_window)
            ax.plot(x, mean_arr, lw=1, color=color, label=system)
            ax.fill_between(x, mean_arr - err_arr, mean_arr + err_arr, color=color, alpha=0.20, linewidth=0)
    if kind == "rmsf":
        ax.set_xlabel(xlabel or axis_label)
        ax.set_ylabel(ylabel or "RMSF (Å)")
    else:
        ax.set_xlabel(xlabel or "Time (ns)")
        ax.set_ylabel(ylabel or "RMSD (Å)")
    ax.set_title(title or record["label"])
    ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def _update_rms_metric_options(*_):
    kind = _rms_kind_dd.value
    records = [r for r in _RMS_CATALOG if r["kind"] == kind]
    opts = [(r["label"], r["key"]) for r in records]
    previous = _rms_metric_dd.value
    _rms_metric_dd.options = opts
    if opts:
        values = [v for _, v in opts]
        _rms_metric_dd.value = previous if previous in values else values[0]
        _rms_msg.value = f"<span style='font-size:11px;color:#aaa'>{len(opts)} {kind.upper()} file type(s) found in RMSD folders for selected systems.</span>"
    else:
        _rms_metric_dd.value = None
        _rms_msg.value = f"<span style='color:orange'>No {kind.upper()} files found in RMSD folders for selected systems.</span>"
    if callable(globals().get("_update_chart_type_options")):
        _update_chart_type_options()


def _plot_rms_bar(
    record, systems, color_map, *, root, title, xlabel, ylabel, xlim, ylim, fig_size, error_mode, show_dots, individual=False
):
    """Bar chart: mean RMS value per system (averaged over time for RMSD, or over residues for RMSF)."""
    kind, filename = record["kind"], record["filename"]
    if fig_size is None:
        fig_size = (max(4, len(systems) * 0.75 + 0.5), 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    if individual:
        labels, vals, colors = [], [], []
        for system in systems:
            runs = discover_rms_runs(system, root)
            base_color = color_map.get(system, "#888")
            shades = _run_shades(system, runs, base_color)
            for run in runs:
                df = _load_rms_run(system, run, filename, kind, root)
                if df is None or df.empty:
                    continue
                labels.append(f"{system} / {run}")
                vals.append(float(np.nanmean(df["value_a"].to_numpy(dtype=float))))
                colors.append(shades.get(run, base_color))
        x_pos = np.arange(len(labels))
        ax.bar(x_pos, vals, color=colors, alpha=0.85)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=30, ha="right")
        y_label = "RMSF (Å)" if kind == "rmsf" else "RMSD (Å)"
        ax.set_ylabel(ylabel or f"Run mean {y_label}")
        ax.set_xlabel(xlabel or "")
        ax.set_title(title or f"{record['label']} — Individual Run Means")
        ax.spines[["top", "right"]].set_visible(False)
        _finalize(ax, None, xlabel, ylabel, xlim, ylim)
        return fig
    x_pos = np.arange(len(systems))
    means, errs, run_val_sets = [], [], []
    for system in systems:
        vals = []
        for run in discover_rms_runs(system, root):
            df = _load_rms_run(system, run, filename, kind, root)
            if df is not None and not df.empty:
                vals.append(float(np.nanmean(df["value_a"].to_numpy(dtype=float))))
        m = float(np.mean(vals)) if vals else 0.0
        n = len(vals)
        s = float(np.std(vals, ddof=1)) if n > 1 else 0.0
        means.append(m)
        errs.append((1.96 * s / n ** 0.5) if error_mode == "95% CI" else (s / n ** 0.5) if n > 1 else 0.0)
        run_val_sets.append(vals)
    colors = [color_map.get(s, "#888") for s in systems]
    ax.bar(x_pos, means, yerr=errs, color=colors, capsize=4, alpha=0.85,
           error_kw={"elinewidth": 1.2, "capthick": 1.2})
    if show_dots:
        rng = np.random.default_rng(42)
        for xi, vals in zip(x_pos, run_val_sets):
            if vals:
                jitter = rng.uniform(-0.18, 0.18, len(vals))
                ax.scatter(xi + jitter, vals, color="k", s=22, zorder=5, alpha=0.65, linewidths=0)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(systems, rotation=30, ha="right")
    y_label = "RMSF (Å)" if kind == "rmsf" else "RMSD (Å)"
    ax.set_ylabel(ylabel or f"Mean {y_label}")
    ax.set_xlabel(xlabel or "")
    ax.set_title(title or f"{record['label']} — Mean")
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def _plot_rms_kde(
    record, systems, color_map, *, root, title, xlabel, ylabel, xlim, ylim, fig_size, individual=False
):
    """KDE of RMS values across the full trajectory (RMSD) or all residues (RMSF)."""
    kind, filename = record["kind"], record["filename"]
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    plotted = False
    for system in systems:
        runs = discover_rms_runs(system, root)
        color = color_map.get(system, "#888")
        if individual:
            shades = _run_shades(system, runs, color)
            for run in runs:
                df = _load_rms_run(system, run, filename, kind, root)
                if df is None or df.empty:
                    continue
                vals = df["value_a"].dropna().to_numpy(dtype=float)
                vals = vals[np.isfinite(vals)]
                if len(vals) < 5:
                    continue
                sns.kdeplot(data=vals, ax=ax, color=shades.get(run, color), label=f"{system} / {run}", fill=True, alpha=0.08, linewidth=1.2)
                plotted = True
        else:
            all_vals: list[float] = []
            for run in runs:
                df = _load_rms_run(system, run, filename, kind, root)
                if df is not None and not df.empty:
                    all_vals.extend(df["value_a"].dropna().tolist())
            if not all_vals:
                continue
            sns.kdeplot(data=all_vals, ax=ax, color=color, label=system, fill=True, alpha=0.15, linewidth=1.6)
            plotted = True
    x_label = "RMSF (Å)" if kind == "rmsf" else "RMSD (Å)"
    ax.set_xlabel(xlabel or x_label)
    ax.set_ylabel(ylabel or "Density")
    ax.set_title(title or f"{record['label']} — KDE")
    if plotted:
        ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def _run_rms_plot(*_):
    systems = _get_selected_systems()
    record  = next((r for r in _RMS_CATALOG if r["key"] == _rms_metric_dd.value), None)
    if not systems or record is None:
        with _out:
            clear_output(wait=True)
            print("Select systems and an RMSD/RMSF metric first.")
        return
    xlim, ylim, title, xlabel, ylabel = _get_axis_overrides()
    _apply_theme(_theme_dd.value, _palette_dd.value)
    color_map = _get_color_map(systems)
    chart = _chart_type.value if "_chart_type" in globals() else "Time Series"
    with _out:
        clear_output(wait=True)
        plt.close("all")
        if chart == "Bar (mean)":
            fig = _plot_rms_bar(
                record, systems, color_map,
                root=RESULTS_ROOT, title=title, xlabel=xlabel, ylabel=ylabel,
                xlim=xlim, ylim=ylim, fig_size=_get_fig_size(),
                error_mode=_error_mode_dd.value, show_dots=_dots_cb.value,
                individual=_is_individual(),
            )
        elif chart == "KDE":
            fig = _plot_rms_kde(
                record, systems, color_map,
                root=RESULTS_ROOT, title=title, xlabel=xlabel, ylabel=ylabel,
                xlim=xlim, ylim=ylim, fig_size=_get_fig_size(),
                individual=_is_individual(),
            )
        else:  # Time Series / Atom Series
            fig = plot_rms_profile(
                record, systems, color_map,
                individual=_is_individual(),
                root=RESULTS_ROOT,
                smooth_window=int(_rms_smooth.value),
                title=title, xlabel=xlabel, ylabel=ylabel,
                xlim=xlim, ylim=ylim, fig_size=_get_fig_size(),
            )
        if _rms_thresholds_cb.value:
            lo, hi = _get_effective_threshold()
            orient = "v" if chart == "KDE" else "h"
            for ax in fig.get_axes():
                _add_thresh_marker(ax, lo, hi, orient)
        _show_fig(fig)

In [16]:
# RDF UI Widgets

def _rdf_center_options(catalog: list[dict]) -> list[tuple[str, str]]:
    centers = sorted({r["center"] for r in catalog}, key=lambda c: _RDF_CENTER_LABELS.get(c, c))
    return [(_RDF_CENTER_LABELS.get(c, c.replace("_", " ").title()), c) for c in centers]


def _rdf_type_label(rdf_type: str) -> str:
    if rdf_type in {"atp", "mg", "ptk"}:
        return rdf_type.upper()
    return rdf_type.replace("_", " ").title()


def _rdf_type_sort_key(rdf_type: str) -> tuple[int, str]:
    try:
        return (_RDF_TYPE_ORDER.index(rdf_type), rdf_type)
    except ValueError:
        return (len(_RDF_TYPE_ORDER), rdf_type)


def _rdf_type_options(center: str) -> list[tuple[str, str]]:
    types = {r["type"] for r in _RDF_CATALOG if r["center"] == center}
    if not types:
        return []
    options = [("All", "all")]
    options.extend((_rdf_type_label(t), t) for t in sorted(types - {"all"}, key=_rdf_type_sort_key))
    return options


def _update_rdf_type_options(*_):
    center = _rdf_center_dd.value
    if not center:
        _rdf_type_dd.options = []
        _rdf_detail_dd.options = []
        return
    current = _rdf_type_dd.value
    opts = _rdf_type_options(center)
    _rdf_type_dd.options = opts
    values = {value for _, value in opts}
    if current in values:
        _rdf_type_dd.value = current
    elif opts:
        _rdf_type_dd.value = opts[0][1]
    else:
        _rdf_type_dd.value = None
    _update_rdf_detail_options()


def _update_rdf_detail_options(*_):
    center = _rdf_center_dd.value
    rdf_type = _rdf_type_dd.value
    detail = _rdf_view.value == "Individual residues"
    _rdf_detail_dd.layout.display = "" if detail else "none"
    if not center or not rdf_type:
        _rdf_detail_dd.options = []
        return
    if detail:
        records = [
            r for r in _RDF_CATALOG
            if r["center"] == center and (rdf_type == "all" or r.get("type") == rdf_type)
        ]
        opts = [(_rdf_record_display(r), r["key"]) for r in records]
        current = set(_rdf_detail_dd.value or [])
        values = tuple(value for _, value in opts)
        _rdf_detail_dd.options = opts
        kept = tuple(value for value in values if value in current)
        _rdf_detail_dd.value = kept or values
        _rdf_msg.value = "" if opts else "<span style='color:orange'>No RDF files found for this center/type.</span>"
    else:
        records = [r for r in _RDF_CATALOG if r["center"] == center and _rdf_matches_type(r, rdf_type, False)]
        _rdf_detail_dd.options = []
        _rdf_detail_dd.value = tuple()
        _rdf_msg.value = f"<span style='font-size:11px;color:#aaa'>{len(records)} aggregate RDF curve(s) match this center/type.</span>"


def _selected_rdf_records() -> list[dict]:
    center = _rdf_center_dd.value
    rdf_type = _rdf_type_dd.value
    detail = _rdf_view.value == "Individual residues"
    if not center or not rdf_type:
        return []
    if detail:
        selected = set(_rdf_detail_dd.value or [])
        return [
            r for r in _RDF_CATALOG
            if r["key"] in selected and r["center"] == center and (rdf_type == "all" or r.get("type") == rdf_type)
        ]
    return [r for r in _RDF_CATALOG if r["center"] == center and _rdf_matches_type(r, rdf_type, False)]

In [17]:
# Plot Dispatchers

_last_fig: plt.Figure | None = None


def _enforce_axes_size(fig: plt.Figure, w_in: float, h_in: float) -> None:
    """Resize the figure so the primary axes plot area is exactly w_in x h_in inches."""
    axes = fig.get_axes()
    if not axes:
        return
    ax = axes[0]
    bbox = ax.get_position()
    if bbox.width > 0 and bbox.height > 0:
        fig.set_size_inches(w_in / bbox.width, h_in / bbox.height)


def _show_fig(fig):
    global _last_fig
    if fig is None:
        return
    _last_fig = fig
    if not _fig_auto.value:
        _enforce_axes_size(fig, _fig_w.value, _fig_h.value)
    import io, base64
    from IPython.display import HTML, display as _ipy_display
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=_dpi_it.value, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode("ascii")
    _ipy_display(HTML(
        f"<img src='data:image/png;base64,{b64}' "
        f"style='max-width:1000px;max-height:500px;display:block' />"
    ))
    plt.close(fig)


def _run_plot(*_):
    _status.value = ""
    systems = _get_selected_systems()
    _use_leg = False
    if not systems:
        _status.value = "<span style='color:tomato'>Select at least one system.</span>"
        return
    metric = _metric_dd.value
    if not metric:
        with _out:
            clear_output(wait=True)
            print("Select a metric from the dropdown.")
        return
    individual = _is_individual()
    xlim, ylim, title, xlabel, ylabel = _get_axis_overrides()
    _apply_theme(_theme_dd.value, _palette_dd.value)
    color_map = _get_color_map(systems)
    meta = _get_shared_meta(systems)
    mt = _mtype(metric, meta)
    chart = _chart_type.value
    grouped = callable(globals().get("_is_rdf_charge_group_value")) and _is_rdf_charge_group_value(metric)
    with _out:
        clear_output(wait=True)
        plt.close("all")
        if chart == _THRESHOLD_CHART_LABEL:
            if mt not in {"distance", "coordination", "angle"}:
                print("Threshold % is available for distance, angle, and coordination metrics.")
                return
            lo, hi = _threshold_bounds_for_metric(metric, meta)
            if lo is None and hi is None:
                print("No threshold is configured for this metric. Enable threshold override or choose a metric with YAML thresholds.")
                return
            if not _use_ylim_cb.value:
                ylim = (0.0, 100.0)
            label = _rdf_charge_group_title(metric) if grouped else _metric_dropdown_label(metric, meta)
            default_title = f"{label} - trajectory within threshold"
            if individual:
                rows = _collect_threshold_pct_individual(systems, metric, lo, hi, RESULTS_ROOT)
                if not rows:
                    print("No threshold occupancy data found for selected systems/runs.")
                    return
                fig = plot_bar_individual("threshold_pct", rows, color_map, title=title or default_title, xlabel=xlabel, ylabel=ylabel or "% Trajectory", xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), meta=meta, use_legend=_use_leg)
            else:
                st = _collect_threshold_pct_stats(systems, metric, lo, hi, RESULTS_ROOT)
                if not st:
                    print("No threshold occupancy data found for selected systems/runs.")
                    return
                fig = plot_bar("threshold_pct", systems, st, color_map, title=title or default_title, show_dots=_dots_cb.value, error_mode=_error_mode_dd.value, xlabel=xlabel, ylabel=ylabel or "% Trajectory", xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), meta=meta, use_legend=_use_leg)
            _show_fig(fig)
            return
        if grouped:
            metrics = _rdf_charge_group_members(metric, systems, meta)
            if not metrics:
                print("No positive/negative RDF contact metrics found for this grouped selection.")
                return
            group_title = title or _rdf_charge_group_title(metric)
            default_ylabel = "% frames with H-bond" if mt == "hbond" else "Distance (Å)"
            if chart == "Bar (mean)":
                fig = _plot_rdf_charge_group_bar(metrics, systems, color_map, mt, individual=individual, root=RESULTS_ROOT, title=group_title, xlabel=xlabel, ylabel=ylabel or default_ylabel, show_dots=_dots_cb.value, error_mode=_error_mode_dd.value, xlim=xlim, ylim=ylim, fig_size=_get_fig_size())
            elif chart == "KDE":
                if mt == "hbond":
                    print("KDE is not meaningful for binary H-bond data. Use Bar.")
                    return
                fig = _plot_rdf_charge_group_kde(metrics, systems, color_map, mt, individual=individual, root=RESULTS_ROOT, title=group_title, xlabel=xlabel, ylabel=ylabel, xlim=xlim, ylim=ylim, fig_size=_get_fig_size())
                if _show_thresh_cb.value:
                    lo, hi = _get_effective_threshold()
                    for ax in fig.get_axes():
                        _add_thresh_marker(ax, lo, hi, "v")
            elif chart == "Time Series":
                fig = _plot_rdf_charge_group_timeseries(metrics, systems, color_map, mt, individual=individual, root=RESULTS_ROOT, error_mode=_error_mode_dd.value, smooth_window=int(_ts_smooth.value), title=group_title, xlabel=xlabel, ylabel=ylabel or default_ylabel, xlim=xlim, ylim=ylim, fig_size=_get_fig_size())
                if _show_thresh_cb.value:
                    lo, hi = _get_effective_threshold()
                    for ax in fig.get_axes():
                        _add_thresh_marker(ax, lo, hi, "h")
            else:
                print(f"Unsupported chart type: {chart}")
                return
        elif chart == "Bar (mean)":
            if individual:
                rows = collect_individual(systems, None, RESULTS_ROOT)
                if not rows:
                    print("No data found for selected systems/runs.")
                    return
                fig = plot_bar_individual(metric, rows, color_map, title=title, xlabel=xlabel, ylabel=ylabel or _MTYPE_YLABEL.get(mt, "Value"), xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), meta=meta, use_legend=_use_leg)
            else:
                st = collect_stats(systems, None, RESULTS_ROOT)
                if not st:
                    print("No data found for selected systems/runs.")
                    return
                fig = plot_bar(metric, systems, st, color_map, title=title, show_dots=_dots_cb.value, error_mode=_error_mode_dd.value, xlabel=xlabel, ylabel=ylabel or _MTYPE_YLABEL.get(mt, "Value"), xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), meta=meta, use_legend=_use_leg)
            _show_fig(fig)
        elif chart == "KDE":
            if mt == "hbond":
                print("KDE is not meaningful for binary H-bond data. Use Bar.")
                return
            fig = plot_kde(metric, systems, color_map, individual=individual, root=RESULTS_ROOT, title=title, xlabel=xlabel or _MTYPE_YLABEL.get(mt, "Value"), ylabel=ylabel or "Density", xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), meta=meta)
            if _show_thresh_cb.value:
                lo, hi = _get_effective_threshold()
                for ax in fig.get_axes():
                    _add_thresh_marker(ax, lo, hi, "v")
            _show_fig(fig)
        elif chart == "Time Series":
            fig = plot_timeseries(metric, systems, color_map, individual=individual, root=RESULTS_ROOT, error_mode=_error_mode_dd.value, smooth_window=int(_ts_smooth.value), title=title, xlabel=xlabel, ylabel=ylabel or _MTYPE_YLABEL.get(mt, "Value"), xlim=xlim, ylim=ylim, fig_size=_get_fig_size(), meta=meta)
            if _show_thresh_cb.value:
                lo, hi = _get_effective_threshold()
                for ax in fig.get_axes():
                    _add_thresh_marker(ax, lo, hi, "h")
            _show_fig(fig)


def _run_rdf_plot(*_):
    systems = _get_selected_systems()
    records = _selected_rdf_records()
    if not systems or not records:
        with _out:
            clear_output(wait=True)
            print("Select systems and RDF curves first.")
        return
    xlim, ylim, title, xlabel, ylabel = _get_axis_overrides()
    _apply_theme(_theme_dd.value, _palette_dd.value)
    with _out:
        clear_output(wait=True)
        plt.close("all")
        fig = plot_rdfs(records, systems, _get_color_map(systems), individual=_is_individual(), root=RESULTS_ROOT, title=title, xlabel=xlabel, ylabel=ylabel, xlim=xlim, ylim=ylim, fig_size=_get_fig_size())
        if _rdf_thresholds_cb.value:
            lo, hi = _get_effective_threshold()
            for ax in fig.get_axes():
                _add_thresh_marker(ax, lo, hi, "v")
        _show_fig(fig)


def _sync_fig_boxes(*_):
    _fig_w.layout.display = "none" if _fig_auto.value else ""
    _fig_h.layout.display = "none" if _fig_auto.value else ""


def _add_class(widget):
    try:
        widget.add_class("cmp-ui")
    except Exception:
        pass
    if hasattr(widget, "children"):
        for child in widget.children:
            _add_class(child)


def _refresh_rdf_options(*_, force: bool = False):
    global _RDF_CATALOG
    systems = tuple(_get_selected_systems())
    key = (str(RESULTS_ROOT), systems)
    if force or key not in _RDF_CATALOG_CACHE:
        _RDF_CATALOG_CACHE[key] = discover_rdf_catalog(list(systems), RESULTS_ROOT)
    _RDF_CATALOG = _RDF_CATALOG_CACHE[key]
    prev_center = _rdf_center_dd.value
    center_opts = _rdf_center_options(_RDF_CATALOG)
    _rdf_center_dd.options = center_opts
    if center_opts:
        values = [v for _, v in center_opts]
        _rdf_center_dd.value = prev_center if prev_center in values else values[0]
        _update_rdf_type_options()
    else:
        _rdf_type_dd.options = []
        _rdf_detail_dd.options = []
        _rdf_msg.value = "<span style='color:orange'>No RDF files found for selected systems.</span>"


def _refresh_rms_options(*_, force: bool = False):
    global _RMS_CATALOG
    systems = tuple(_get_selected_systems())
    key = (str(RESULTS_ROOT), systems)
    if force or key not in _RMS_CATALOG_CACHE:
        _RMS_CATALOG_CACHE[key] = discover_rms_records(list(systems), RESULTS_ROOT)
    _RMS_CATALOG = _RMS_CATALOG_CACHE[key]
    _update_rms_metric_options()


def _do_system_dependent_update() -> None:
    _rebuild_color_pickers()
    _schedule_metric_update()
    _refresh_rdf_options()
    _refresh_rms_options()


In [18]:
# 2D Scatter Metrics

_SCATTER_MODE_OPTIONS = [("Frame samples", "frames"), ("Run means", "means")]

_rdf_thresholds_cb = widgets.Checkbox(value=False, description="Show threshold shading")
_rms_thresholds_cb = widgets.Checkbox(value=False, description="Show threshold shading")


_scatter_x_dd = widgets.Dropdown(options=[], description="X metric:")
_scatter_y_dd = widgets.Dropdown(options=[], description="Y metric:")
_scatter_x_search_wt = widgets.Text(placeholder="Search X metrics...", description="X search:")
_scatter_y_search_wt = widgets.Text(placeholder="Search Y metrics...", description="Y search:")
_scatter_mode = widgets.ToggleButtons(options=_SCATTER_MODE_OPTIONS, value="frames")
_scatter_max_points = widgets.BoundedIntText(value=1200, min=50, max=50000, step=50, description="Max points/run:")
_scatter_alpha = widgets.BoundedFloatText(value=0.50, min=0.05, max=1.0, step=0.05, description="Alpha:")
_scatter_means_cb = widgets.Checkbox(value=True, description="Overlay run means")
_scatter_thresholds_x_cb = widgets.Checkbox(value=False, description="Show threshold shading X")
_scatter_thresholds_y_cb = widgets.Checkbox(value=False, description="Show threshold shading Y")
_scatter_thresh_x_lo_wt = widgets.BoundedFloatText(value=0.0, min=-9999.0, max=9999.0, step=0.1, description="X Low:")
_scatter_thresh_x_hi_wt = widgets.BoundedFloatText(value=0.0, min=-9999.0, max=9999.0, step=0.1, description="X High:")
_scatter_thresh_y_lo_wt = widgets.BoundedFloatText(value=0.0, min=-9999.0, max=9999.0, step=0.1, description="Y Low:")
_scatter_thresh_y_hi_wt = widgets.BoundedFloatText(value=0.0, min=-9999.0, max=9999.0, step=0.1, description="Y High:")
_scatter_thresh_x_box = widgets.HBox([_scatter_thresh_x_lo_wt, _scatter_thresh_x_hi_wt])
_scatter_thresh_y_box = widgets.HBox([_scatter_thresh_y_lo_wt, _scatter_thresh_y_hi_wt])
_scatter_thresh_vbox = widgets.VBox([_scatter_thresh_x_box, _scatter_thresh_y_box])
_scatter_help = widgets.HTML("<span style='font-size:11px;color:#777'>Pick X/Y metrics to explore relationships per-frame or per-run means.</span>")
_scatter_info = widgets.HTML("")
_scatter_pane = widgets.VBox([
    _scatter_help,
    _scatter_x_search_wt,
    _scatter_x_dd,
    _scatter_y_search_wt,
    _scatter_y_dd,
    _scatter_info,
])


def _scatter_metric_options(systems: list[str], search: str = "") -> list[tuple[str, str]]:
    if not systems:
        return []
    meta = _get_shared_meta(systems)
    metrics = all_metric_names(collect_stats(systems, None, RESULTS_ROOT))
    filtered = [m for m in metrics if not (callable(globals().get("_is_rdf_charge_group_value")) and _is_rdf_charge_group_value(m))]
    term = (search or "").strip().lower()
    if term:
        filtered = [m for m in filtered if term in _metric_search_text(m, meta)]
    return [
        ("  " + _metric_dropdown_label(metric, meta), metric)
        for metric in sorted(filtered, key=lambda m: (_MT_SORT_ORDER.get(_mtype(m, meta), 99), _metric_dropdown_label(m, meta).lower(), m))
    ]


def _preferred_scatter_pair(options: list[tuple[str, str]]) -> tuple[str | None, str | None]:
    values = [value for _, value in options]
    preferred_x = next((m for m in values if m == "cat_arg_from_coordination_atp_neg_gamma_phosphate_dist"), None)
    preferred_y = next((m for m in values if m == "pep_Y_atp_2_angle"), None)
    if preferred_x is None:
        preferred_x = values[0] if values else None
    if preferred_y is None:
        preferred_y = values[1] if len(values) > 1 else preferred_x
    if preferred_x == preferred_y and len(values) > 1:
        preferred_y = next((m for m in values if m != preferred_x), preferred_y)
    return preferred_x, preferred_y


def _update_scatter_metric_options(*_) -> None:
    systems = _get_selected_systems()
    prev_x = _scatter_x_dd.value
    prev_y = _scatter_y_dd.value
    opts_x = _scatter_metric_options(systems, _scatter_x_search_wt.value)
    opts_y = _scatter_metric_options(systems, _scatter_y_search_wt.value)
    _scatter_x_dd.options = opts_x
    _scatter_y_dd.options = opts_y
    if not opts_x or not opts_y:
        _scatter_info.value = ""
        return
    values_x = [value for _, value in opts_x]
    values_y = [value for _, value in opts_y]
    all_opts = _scatter_metric_options(systems, "")
    preferred_x, preferred_y = _preferred_scatter_pair(all_opts)
    default_x = preferred_x if preferred_x in values_x else values_x[0]
    default_y = preferred_y if preferred_y in values_y else values_y[0]
    chosen_x = prev_x if prev_x in values_x else default_x
    chosen_y = prev_y if prev_y in values_y else default_y
    if chosen_x == chosen_y and len(values_y) > 1:
        chosen_y = next((v for v in values_y if v != chosen_x), values_y[0])
    _scatter_x_dd.value = chosen_x
    _scatter_y_dd.value = chosen_y
    _update_scatter_info()


def _scatter_axis_label(metric: str, meta: dict | None = None) -> str:
    label = _clean_metric_label_text((_meta_row_for_metric(metric, meta) or {}).get("display_name")) or metric.replace("_", " ")
    mt = _mtype(metric, meta)
    unit = _MTYPE_UNIT.get(mt, "")
    return f"{label} ({unit})" if unit else label


def _update_scatter_info(*_) -> None:
    systems = _get_selected_systems()
    meta = _get_shared_meta(systems)
    x_metric = _scatter_x_dd.value
    y_metric = _scatter_y_dd.value
    if not x_metric or not y_metric:
        _scatter_info.value = ""
        return
    x_type = _mtype(x_metric, meta)
    y_type = _mtype(y_metric, meta)
    _scatter_info.value = (
        "<span style='font-size:11px;color:#aaa'>"
        f"x: <b>{x_type}</b> &nbsp; y: <b>{y_type}</b> &nbsp; "
        f"mode: <b>{dict(_SCATTER_MODE_OPTIONS).get(_scatter_mode.value, _scatter_mode.value)}</b>"
        "</span>"
    )


def _paired_metric_arrays(system: str, run: str, x_metric: str, y_metric: str, root: Path = RESULTS_ROOT) -> tuple[np.ndarray, np.ndarray] | None:
    x_arr = load_run_raw(system, run, x_metric, root)
    y_arr = load_run_raw(system, run, y_metric, root)
    if x_arr is None or y_arr is None:
        return None
    n = min(len(x_arr), len(y_arr))
    if n == 0:
        return None
    x = np.asarray(x_arr[:n], dtype=float)
    y = np.asarray(y_arr[:n], dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if not np.any(mask):
        return None
    return x[mask], y[mask]


def _sample_xy(x: np.ndarray, y: np.ndarray, max_points: int, seed_key: str) -> tuple[np.ndarray, np.ndarray]:
    if len(x) <= max_points:
        return x, y
    seed = abs(hash(seed_key)) % (2 ** 32)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(x), size=max_points, replace=False))
    return x[idx], y[idx]


def plot_metric_scatter(x_metric: str, y_metric: str, systems: list[str], color_map: dict[str, str], *,
                        individual: bool = False, root: Path = RESULTS_ROOT, mode: str = "frames",
                        max_points: int = 1200, alpha: float = 0.35, overlay_means: bool = True,
                        show_thresholds_x: bool = True, show_thresholds_y: bool = True, title: str | None = None, xlabel: str | None = None,
                        ylabel: str | None = None, xlim=None, ylim=None, fig_size: tuple | None = None,
                        meta: dict | None = None, x_thresh: tuple[float | None, float | None] | None = None,
                        y_thresh: tuple[float | None, float | None] | None = None) -> plt.Figure:
    if fig_size is None:
        fig_size = (4, 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    plotted = False
    run_mean_rows: list[tuple[str, str, float, float]] = []
    marker_cycle = ["o", "s", "^", "D", "P", "X", "v", "<", ">"]
    for system in systems:
        runs = discover_runs(system, root)
        color = color_map.get(system, "#888")
        shades = _run_shades(system, runs, color)
        for run_index, run in enumerate(runs):
            paired = _paired_metric_arrays(system, run, x_metric, y_metric, root)
            if paired is None:
                continue
            x, y = paired
            x_mean = float(np.nanmean(x))
            y_mean = float(np.nanmean(y))
            run_mean_rows.append((system, run, x_mean, y_mean))
            if mode == "means":
                marker = marker_cycle[run_index % len(marker_cycle)] if individual else "o"
                label = f"{system} / {run}" if individual else None
                ax.scatter([x_mean], [y_mean], s=58, color=shades.get(run, color) if individual else color,
                           marker=marker, edgecolor="white", linewidth=0.6, alpha=0.95, label=label or system, zorder=4)
                plotted = True
            else:
                xs, ys = _sample_xy(x, y, int(max_points), f"{system}|{run}|{x_metric}|{y_metric}")
                label = f"{system} / {run}" if individual else (system if run_index == 0 else None)
                ax.scatter(xs, ys, s=14, color=shades.get(run, color) if individual else color,
                           alpha=float(alpha), linewidths=0, label=label, rasterized=True)
                plotted = True
    if overlay_means and mode == "frames" and run_mean_rows:
        for system in systems:
            color = color_map.get(system, "#888")
            rows = [(run, x, y) for row_system, run, x, y in run_mean_rows if row_system == system]
            for run, x_mean, y_mean in rows:
                ax.scatter([x_mean], [y_mean], s=72, color=color, edgecolor="black", linewidth=0.8, marker="X", zorder=5)
    if meta is not None:
        if show_thresholds_x:
            x_lo, x_hi = x_thresh if x_thresh is not None else _thresh_for(x_metric, meta)
            _add_thresh_marker(ax, x_lo, x_hi, "v")
        if show_thresholds_y:
            y_lo, y_hi = y_thresh if y_thresh is not None else _thresh_for(y_metric, meta)
            _add_thresh_marker(ax, y_lo, y_hi, "h")
    ax.set_xlabel(xlabel or _scatter_axis_label(x_metric, meta))
    ax.set_ylabel(ylabel or _scatter_axis_label(y_metric, meta))
    ax.set_title(title or f"{x_metric} vs {y_metric}")
    if plotted:
        handles, labels = ax.get_legend_handles_labels()
        dedup = dict(zip(labels, handles))
        ax.legend(dedup.values(), dedup.keys(), loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, None, xlabel, ylabel, xlim, ylim)
    return fig


def _run_scatter_plot(*_) -> None:
    systems = _get_selected_systems()
    x_metric = _scatter_x_dd.value
    y_metric = _scatter_y_dd.value
    if not systems or not x_metric or not y_metric:
        with _out:
            clear_output(wait=True)
            print("Select systems plus X and Y metrics first.")
        return
    if x_metric == y_metric:
        _status.value = "<span style='color:orange'>Choose two different metrics for a 2D scatter.</span>"
        return
    xlim, ylim, title, xlabel, ylabel = _get_axis_overrides()
    meta = _get_shared_meta(systems)
    _apply_theme(_theme_dd.value, _palette_dd.value)
    with _out:
        clear_output(wait=True)
        plt.close("all")
        fig = plot_metric_scatter(
            x_metric, y_metric, systems, _get_color_map(systems),
            individual=_is_individual(), root=RESULTS_ROOT, mode=_scatter_mode.value,
            max_points=int(_scatter_max_points.value), alpha=float(_scatter_alpha.value),
            overlay_means=bool(_scatter_means_cb.value),
            show_thresholds_x=bool(_scatter_thresholds_x_cb.value), show_thresholds_y=bool(_scatter_thresholds_y_cb.value),
            title=title, xlabel=xlabel, ylabel=ylabel, xlim=xlim, ylim=ylim,
            fig_size=_get_fig_size(), meta=meta,
            x_thresh=(float(_scatter_thresh_x_lo_wt.value), float(_scatter_thresh_x_hi_wt.value)),
            y_thresh=(float(_scatter_thresh_y_lo_wt.value), float(_scatter_thresh_y_hi_wt.value)),
        )
        _show_fig(fig)


def _show_threshold_bounds_for_chart(show_thresholds: bool, chart_type: str) -> bool:
    return bool(show_thresholds or chart_type == _THRESHOLD_CHART_LABEL)


def _set_widget_visible(widget, visible: bool) -> None:
    widget.layout.display = "" if visible else "none"


def _sync_scatter_threshold_boxes() -> None:
    _set_widget_visible(_scatter_thresh_x_box, bool(_scatter_thresholds_x_cb.value))
    _set_widget_visible(_scatter_thresh_y_box, bool(_scatter_thresholds_y_cb.value))
    _set_widget_visible(_scatter_thresh_vbox, bool(_scatter_thresholds_x_cb.value or _scatter_thresholds_y_cb.value))


def _rebuild_chart_sub_options(*_):
    ct = _chart_type.value
    idx = getattr(_data_tabs, "selected_index", 0) if "_data_tabs" in globals() else 0
    if idx == 5:
        _chart_sub_box.children = []
        return
    if idx == 4:
        _chart_sub_box.children = [_error_mode_dd, _dots_cb]
        return
    if idx == 3:
        _sync_scatter_threshold_boxes()
        _chart_sub_box.children = [
            widgets.HBox([_scatter_mode, _scatter_max_points, _scatter_alpha]),
            widgets.VBox([_scatter_means_cb, _scatter_thresholds_x_cb, _scatter_thresholds_y_cb]),
            _scatter_thresh_vbox,
        ]
        return
    if idx == 2:
        _set_widget_visible(_thresh_override_box, bool(_rms_thresholds_cb.value))
        if ct == "Bar (mean)":
            _chart_sub_box.children = [_error_mode_dd, _dots_cb, _rms_thresholds_cb, _thresh_override_box]
        elif ct == "KDE":
            _chart_sub_box.children = [_rms_thresholds_cb, _thresh_override_box]
        else:
            _chart_sub_box.children = [_error_mode_dd, _rms_smooth, _rms_thresholds_cb, _thresh_override_box]
        return
    if idx == 1:
        _set_widget_visible(_thresh_override_box, bool(_rdf_thresholds_cb.value))
        _chart_sub_box.children = [_error_mode_dd, _rdf_thresholds_cb, _thresh_override_box]
        return
    col = _metric_dd.value
    mt = _mtype(col, _get_shared_meta(_get_selected_systems())) if col else "distance"
    _set_widget_visible(_thresh_override_box, _show_threshold_bounds_for_chart(_show_thresh_cb.value, ct))
    if ct == "Bar (mean)":
        _chart_sub_box.children = [_error_mode_dd, _dots_cb]
    elif ct == "KDE":
        if mt == "hbond":
            _chart_sub_box.children = [widgets.HTML("<span style='color:orange'>KDE is not meaningful for binary H-bond data. Use Bar.</span>")]
        else:
            _chart_sub_box.children = [_show_thresh_cb, _thresh_override_box]
    elif ct == "Time Series":
        _chart_sub_box.children = [_error_mode_dd, _ts_smooth, _show_thresh_cb, _thresh_override_box]
    elif ct == _THRESHOLD_CHART_LABEL:
        if mt not in {"distance", "coordination", "angle"}:
            _chart_sub_box.children = [widgets.HTML("<span style='color:orange'>Threshold % is available for distance, angle, and coordination metrics.</span>")]
        else:
            _chart_sub_box.children = [_error_mode_dd, _dots_cb, _thresh_override_box]


In [19]:
# Clustering Analysis

_CLUSTER_DATA_CACHE: dict[str, "pd.DataFrame | None"] = globals().get("_CLUSTER_DATA_CACHE", {})
_CLUSTER_CATALOG: list[dict] = []
_CLUSTER_UNIT_COLOR_MAP: dict[str, str] = globals().get("_CLUSTER_UNIT_COLOR_MAP", {})


_cluster_config_dd = widgets.Dropdown(
    options=[],
    description="Config:",
)
_cluster_group_by = widgets.ToggleButtons(
    options=[("By run", "run"), ("By system", "system"), ("By peptide", "peptide"), ("By kinase", "kinase")],
    value="system",
    description="Group by:",
)
_cluster_chart_type = widgets.ToggleButtons(
    options=[("Grouped bars", "bar"), ("Stacked bars", "stacked"), ("Heatmap", "heatmap")],
    value="bar",
    description="Chart:",
)
_cluster_group_order_wt = widgets.Text(
    value="",
    placeholder="Group order: comma-separated (empty = sorted)",
    description="Group order:",
)
_cluster_cluster_order_wt = widgets.Text(
    value="",
    placeholder="Cluster order: e.g. 0, 1, -1 (empty = sorted)",
    description="Cluster order:",
)
_cluster_palette_dd = widgets.Dropdown(
    options=_PALETTES,
    value="Custom 4-Color",
    description="Palette:",
)
_cluster_color_box = widgets.VBox([])
_cluster_help = widgets.HTML("<span style='font-size:11px;color:#777'>Choose clustering config and grouping to compare cluster populations.</span>")
_cluster_info = widgets.HTML("")
_cluster_details = widgets.HTML("")
_clustering_pane = widgets.VBox([
    _cluster_help,
    _cluster_config_dd,
    _cluster_details,
    widgets.HBox([_cluster_group_by, _cluster_chart_type]),
])


def _discover_clustering_configs(root: "Path" = RESULTS_ROOT) -> list[dict]:
    """Scan results/ for Clustering dirs and return available config entries."""
    entries: list[dict] = []
    seen: set[str] = set()

    def _scan_dir(cluster_dir: "Path", system: "str | None") -> None:
        if not cluster_dir.is_dir():
            return
        for config_dir in sorted(cluster_dir.iterdir()):
            if not config_dir.is_dir():
                continue
            frame_map = config_dir / "frame_map.csv"
            centroid_dir = config_dir / "Centroid_Structures"
            if not centroid_dir.is_dir():
                continue
            label_csvs = sorted(centroid_dir.glob("*_cluster_labels.csv"))
            meta_csvs = sorted(centroid_dir.glob("*_centroids_meta.csv"))
            if not label_csvs:
                continue
            key = str(config_dir.resolve())
            if key in seen:
                continue
            seen.add(key)
            group_stem = label_csvs[0].stem.replace("_cluster_labels", "")
            display_name = config_dir.name if system is None else f"{system} / {config_dir.name}"
            config_yaml = cluster_dir.parent.parent / "config" / config_dir.name
            entries.append({
                "display_name": display_name,
                "config_name": config_dir.name,
                "config_dir": config_dir,
                "frame_map": frame_map,
                "config_yaml": config_yaml if config_yaml.exists() else None,
                "label_csv": label_csvs[0],
                "meta_csv": meta_csvs[0] if meta_csvs else None,
                "system": system,
                "group_stem": group_stem,
            })

    _scan_dir(root / "Clustering", None)
    for sys_dir in sorted(root.iterdir()):
        if sys_dir.is_dir() and sys_dir.name != "Clustering":
            _scan_dir(sys_dir / "Clustering", sys_dir.name)
    return entries


def _load_cluster_frames(entry: dict, root: "Path" = RESULTS_ROOT) -> "pd.DataFrame | None":
    """Load frame_map + cluster_labels merged into a per-frame DataFrame."""
    cache_key = str(entry["config_dir"])
    if cache_key in _CLUSTER_DATA_CACHE:
        return _CLUSTER_DATA_CACHE[cache_key]
    labels = pd.read_csv(entry["label_csv"])
    frame_map_path = entry["frame_map"]
    if frame_map_path.exists():
        frame_map = pd.read_csv(frame_map_path)
        rows: list[dict] = []
        for _, row in frame_map.iterrows():
            for fi in range(int(row["start_frame"]), int(row["end_frame"]) + 1):
                rows.append({
                    "frame_index": fi,
                    "system": str(row["system"]),
                    "run": str(row["run"]),
                })
        fmap_df = pd.DataFrame(rows)
        merged = labels.merge(fmap_df, on="frame_index", how="left")
        merged["system"] = merged["system"].fillna("unknown")
        merged["run"] = merged["run"].fillna("unknown")
    _CLUSTER_DATA_CACHE[cache_key] = merged
    return merged


def _cluster_system_kinase(system: str) -> str:
    return system.split("_", 1)[0]


def _cluster_system_peptide(system: str) -> str:
    parts = system.split("_")
    return parts[1] if len(parts) >= 2 else system


def _build_cluster_details_html(entry: dict) -> str:
    """Return an HTML string summarising what files will be loaded for a config entry."""
    results_root = RESULTS_ROOT
    rel = lambda p: str(Path(p).relative_to(results_root)) if p and Path(p).is_relative_to(results_root) else str(p)

    label_path = entry["label_csv"]
    frame_map_path = entry["frame_map"]
    meta_path = entry.get("meta_csv")
    config_yaml = entry.get("config_yaml")

    label_exists = label_path.exists() if label_path else False
    fm_exists = frame_map_path.exists() if frame_map_path else False

    n_labels = "?"
    n_fm_rows = "?"
    systems_runs = ""
    if label_exists:
        try:
            _ldf = pd.read_csv(label_path)
            n_labels = str(len(_ldf))
        except Exception:
            pass
    if fm_exists:
        try:
            _fdf = pd.read_csv(frame_map_path)
            n_fm_rows = str(len(_fdf))
            sr = _fdf.groupby("system")["run"].apply(list).to_dict()
            systems_runs = "; ".join(
                f"{s}: {', '.join(sorted(set(rs)))}" for s, rs in sr.items()
            )
        except Exception:
            pass

    def _row(label, value, ok=True):
        color = "#ccc" if ok else "tomato"
        return f"<tr><td style='color:#888;padding-right:8px;white-space:nowrap'>{label}</td><td style='color:{color};word-break:break-all'>{value}</td></tr>"

    rows = [
        _row("labels", f"{rel(label_path)} ({n_labels} frames)", label_exists),
        _row("frame_map", f"{rel(frame_map_path)} ({n_fm_rows} segments)" if fm_exists else "<em>not found — will reconstruct</em>", fm_exists),
        _row("systems/runs", systems_runs if systems_runs else "—"),
        _row("meta", rel(meta_path) if meta_path else "—", bool(meta_path)),
    ]
    return (
        "<table style='font-size:10px;font-family:monospace;margin-top:4px;border-collapse:collapse'>"
        + "".join(rows)
        + "</table>"
    )


def _cluster_get_color_map(groups: list[str], group_by: str, df: "pd.DataFrame") -> dict[str, str]:
    """
    Build a color map for clustering groups.

    - run: each unique system gets its base color from _CLUSTER_UNIT_COLOR_MAP / palette,
           then runs within that system get light->dark shades via _run_shades.
    - system/peptide/kinase: use _CLUSTER_UNIT_COLOR_MAP overrides, palette for defaults.
    """
    palette = _cluster_palette_dd.value
    pal_obj = CUSTOM_PALETTES.get(palette, None if palette == "Default" else palette)

    if group_by == "run":
        systems_in_data = sorted(df["system"].unique())
        base_colors = sns.color_palette(pal_obj, n_colors=max(1, len(systems_in_data)))
        sys_base: dict[str, str] = {}
        for i, sys in enumerate(systems_in_data):
            sys_base[sys] = _CLUSTER_UNIT_COLOR_MAP.get(sys, mcolors.to_hex(base_colors[i]))

        color_map: dict[str, str] = {}
        for sys in systems_in_data:
            runs_for_sys = sorted(df[df["system"] == sys]["run"].unique())
            shades = _run_shades(sys, runs_for_sys, sys_base[sys])
            for run in runs_for_sys:
                key = f"{sys} / {run}"
                color_map[key] = shades.get(run, sys_base[sys])
        return color_map
    else:
        base_colors = sns.color_palette(pal_obj, n_colors=max(1, len(groups)))
        return {
            grp: _CLUSTER_UNIT_COLOR_MAP.get(grp, mcolors.to_hex(base_colors[i]))
            for i, grp in enumerate(groups)
        }


def _refresh_clustering_options(force: bool = False) -> None:
    global _CLUSTER_CATALOG
    if force or not _CLUSTER_CATALOG:
        _CLUSTER_CATALOG = _discover_clustering_configs(RESULTS_ROOT)
    opts = [(e["display_name"], i) for i, e in enumerate(_CLUSTER_CATALOG)]
    if not opts:
        _cluster_config_dd.options = []
        _cluster_details.value = ""
        _cluster_color_box.children = []
        _cluster_info.value = "<span style='color:#aaa;font-size:11px'>No clustering results found.</span>"
        return
    prev = _cluster_config_dd.value
    _cluster_config_dd.options = opts
    if prev is not None and prev in {v for _, v in opts}:
        _cluster_config_dd.value = prev
    _cluster_info.value = f"<span style='color:#aaa;font-size:11px'>{len(opts)} config(s) found</span>"


def _rebuild_cluster_color_pickers(groups: list[str], group_by: str, df: "pd.DataFrame | None") -> None:
    """Rebuild per-group color pickers. For run mode shows per-system pickers (shades auto-derived)."""
    if df is None or not groups:
        _cluster_color_box.children = []
        return

    palette = _cluster_palette_dd.value
    pal_obj = CUSTOM_PALETTES.get(palette, None if palette == "Default" else palette)

    if group_by == "run":
        systems_in_data = sorted(df["system"].unique())
        base_colors = sns.color_palette(pal_obj, n_colors=max(1, len(systems_in_data)))
        pickers = []
        for i, sys in enumerate(systems_in_data):
            default_color = mcolors.to_hex(base_colors[i])
            color = _CLUSTER_UNIT_COLOR_MAP.get(sys, default_color)
            _CLUSTER_UNIT_COLOR_MAP.setdefault(sys, color)
            cp = widgets.ColorPicker(
                description=sys, value=color,
            )
            def _on_color(change, _k=sys):
                if change["name"] == "value":
                    _CLUSTER_UNIT_COLOR_MAP[_k] = change["new"]
            cp.observe(_on_color, names="value")
            pickers.append(cp)
        note = widgets.HTML("<span style='font-size:10px;color:#aaa'>Run shades derived automatically from system color</span>")
        _cluster_color_box.children = [
            widgets.HTML("<b>Base colors (runs shaded automatically)</b>"),
            widgets.VBox(pickers),
            note,
        ]
    else:
        base_colors = sns.color_palette(pal_obj, n_colors=max(1, len(groups)))
        pickers = []
        for i, grp in enumerate(groups):
            default_color = mcolors.to_hex(base_colors[i])
            color = _CLUSTER_UNIT_COLOR_MAP.get(grp, default_color)
            _CLUSTER_UNIT_COLOR_MAP.setdefault(grp, color)
            cp = widgets.ColorPicker(
                description=grp, value=color,
            )
            def _on_color(change, _k=grp):
                if change["name"] == "value":
                    _CLUSTER_UNIT_COLOR_MAP[_k] = change["new"]
            cp.observe(_on_color, names="value")
            pickers.append(cp)
        _cluster_color_box.children = [
            widgets.HTML("<b>Series colors</b>"),
            widgets.VBox(pickers),
        ]


def _on_cluster_config_change(change) -> None:
    idx = change["new"]
    if idx is None or not _CLUSTER_CATALOG:
        _cluster_details.value = ""
        _cluster_color_box.children = []
        return
    entry = _CLUSTER_CATALOG[idx]
    _cluster_details.value = _build_cluster_details_html(entry)
    _update_cluster_color_pickers()


def _update_cluster_color_pickers(*_) -> None:
    idx = _cluster_config_dd.value
    if idx is None or not _CLUSTER_CATALOG:
        return
    entry = _CLUSTER_CATALOG[idx]
    df = _load_cluster_frames(entry)
    if df is None:
        _cluster_color_box.children = []
        return
    group_by = _cluster_group_by.value
    df_plot = df.copy()
    if group_by == "run":
        df_plot["group"] = df_plot["system"] + " / " + df_plot["run"]
    elif group_by == "system":
        df_plot["group"] = df_plot["system"]
    elif group_by == "peptide":
        df_plot["group"] = df_plot["system"].apply(_cluster_system_peptide)
    elif group_by == "kinase":
        df_plot["group"] = df_plot["system"].apply(_cluster_system_kinase)
    else:
        df_plot["group"] = df_plot["system"]
    groups = sorted(df_plot["group"].unique())
    _rebuild_cluster_color_pickers(groups, group_by, df)


def _plot_cluster_sizes(
    df: "pd.DataFrame",
    group_by: str,
    chart_type: str,
    config_entry: dict,
    title: "str | None" = None,
    fig_size: "tuple | None" = None,
    group_order: "list[str] | None" = None,
    cluster_order: "list[int] | None" = None,
) -> "plt.Figure":
    """Bar / stacked / heatmap of cluster frame counts broken down by group_by."""
    _apply_font_settings()
    xlim, ylim, title_override, xlabel_override, ylabel_override = _get_axis_overrides()
    if title_override:
        title = title_override

    if df is None or df.empty:
        fig, ax = plt.subplots(figsize=fig_size or (4, 2.5))
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        return fig

    df_plot = df.copy()
    if group_by == "run":
        df_plot["group"] = df_plot["system"] + " / " + df_plot["run"]
    elif group_by == "system":
        df_plot["group"] = df_plot["system"]
    elif group_by == "peptide":
        df_plot["group"] = df_plot["system"].apply(_cluster_system_peptide)
    elif group_by == "kinase":
        df_plot["group"] = df_plot["system"].apply(_cluster_system_kinase)
    else:
        df_plot["group"] = df_plot["system"]

    counts = df_plot.groupby(["cluster_id", "group"]).size().reset_index(name="n_frames")
    _all_groups = counts["group"].unique()
    _all_cluster_ids = counts["cluster_id"].unique()
    groups = [g for g in group_order if g in _all_groups] if group_order else sorted(_all_groups)
    cluster_ids = [c for c in cluster_order if c in _all_cluster_ids] if cluster_order is not None else sorted(_all_cluster_ids)

    color_map = _cluster_get_color_map(groups, group_by, df)
    config_label = config_entry.get("display_name", "")
    auto_title = title or f"Cluster sizes  {config_label}"
    auto_xlabel = xlabel_override or "Cluster"
    auto_ylabel = ylabel_override or "Number of frames"

    if chart_type == "heatmap":
        pivot = counts.pivot(index="group", columns="cluster_id", values="n_frames").fillna(0)
        if fig_size is None:
            fig_size = (max(4, len(cluster_ids) * 0.5 + 0.5), max(2.5, len(groups) * 0.35 + 0.5))
        fig, ax = plt.subplots(figsize=fig_size)
        im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"C{c}" if c >= 0 else "Noise" for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        plt.colorbar(im, ax=ax, label="Frames")
        ax.spines[["top", "right"]].set_visible(False)
        _finalize(ax, auto_title, auto_xlabel, None, xlim, ylim)
        return fig

    x_items = list(cluster_ids)
    bar_items = list(groups)
    def _bar_color(k): return color_map.get(k)
    def _bar_label(k): return k
    x_tick_labels = [f"Cluster {c}" if c >= 0 else "Noise" for c in cluster_ids]
    n_x, n_bars = len(cluster_ids), len(groups)

    if fig_size is None:
        if chart_type == "stacked":
            fig_size = (max(4, n_x * 0.5 + 0.5), 2.5)
        else:
            fig_size = (max(4, n_x * max(0.5, n_bars * 0.5) + 0.5), 2.5)
    fig, ax = plt.subplots(figsize=fig_size)
    x = np.arange(n_x)

    if chart_type == "bar":
        width = min(0.8 / max(n_bars, 1), 0.3)
        for bi, bar_key in enumerate(bar_items):
            vals = []
            for x_key in x_items:
                row = counts[(counts["cluster_id"] == x_key) & (counts["group"] == bar_key)]
                vals.append(int(row["n_frames"].iloc[0]) if not row.empty else 0)
            offset = (bi - (n_bars - 1) / 2) * width
            ax.bar(
                x + offset, vals, width=width * 0.92, label=_bar_label(bar_key),
                color=_bar_color(bar_key), edgecolor="white", linewidth=0.5,
            )
    elif chart_type == "stacked":
        bottom = np.zeros(n_x, dtype=float)
        for bi, bar_key in enumerate(bar_items):
            vals = np.zeros(n_x, dtype=float)
            for xi, x_key in enumerate(x_items):
                row = counts[(counts["cluster_id"] == x_key) & (counts["group"] == bar_key)]
                vals[xi] = int(row["n_frames"].iloc[0]) if not row.empty else 0
            ax.bar(
                x, vals, bottom=bottom, label=_bar_label(bar_key),
                color=_bar_color(bar_key), edgecolor="white", linewidth=0.4,
            )
            bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(
        x_tick_labels,
        rotation=30 if n_x > 4 else 0,
        ha="right" if n_x > 4 else "center",
    )
    if 1 <= n_bars <= 20:
        ax.legend(loc="best", ncols=2, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    _finalize(ax, auto_title, auto_xlabel, auto_ylabel, xlim, ylim)
    return fig


def _run_clustering_plot(*_) -> None:
    _status.value = ""
    idx = _cluster_config_dd.value
    if idx is None or not _CLUSTER_CATALOG:
        _status.value = "<span style='color:tomato'>Select a clustering config first.</span>"
        return
    entry = _CLUSTER_CATALOG[idx]
    df = _load_cluster_frames(entry)
    if df is None:
        _status.value = "<span style='color:tomato'>Could not load cluster data.</span>"
        return
    with _out:
        raw_go = _cluster_group_order_wt.value.strip()
        group_order = [g.strip() for g in raw_go.split(",")] if raw_go else None
        raw_co = _cluster_cluster_order_wt.value.strip()
        cluster_order = None
        if raw_co:
            try:
                cluster_order = [int(c.strip()) for c in raw_co.split(",") if c.strip()]
            except ValueError:
                pass
        clear_output(wait=True)
        fig = _plot_cluster_sizes(
            df=df,
            group_by=_cluster_group_by.value,
            chart_type=_cluster_chart_type.value,
            config_entry=entry,
            group_order=group_order,
            cluster_order=cluster_order,
            fig_size=_get_fig_size(),
            title=None,
        )
        _show_fig(fig)


_cluster_config_dd.observe(_on_cluster_config_change, names="value")
_cluster_group_by.observe(_update_cluster_color_pickers, names="value")
_cluster_palette_dd.observe(_update_cluster_color_pickers, names="value")



In [20]:
# Layout, Callbacks, and Display

_DEFAULT_DATA_TAB_INDEX = 0

_FIXED_WIDTH = "1020px"
_L_LEFT  = widgets.Layout(width="300px", flex="0 0 300px")
_L_MID   = widgets.Layout(width="720px", flex="0 0 720px")
_L_PANEL = widgets.Layout(margin="0 0 8px 0")


def _section_header(title: str, subtitle: str = ""):
    subtitle_html = f"<div class='ui-section-subtitle'>{subtitle}</div>" if subtitle else ""
    return widgets.HTML(
        f"<div class='ui-section-header'>"
        f"<div class='ui-section-title'>{title}</div>"
        f"{subtitle_html}"
        f"</div>"
    )


# ── CSS ───────────────────────────────────────────────────────────────────────
_ui_css = widgets.HTML("""
<style>
/* ── Root typography ─────────────────────────────────────────────────── */
.ui-root,
.ui-root .widget-box,
.ui-root .widget-vbox,
.ui-root .widget-hbox {
  font-family: "Inter", "Avenir Next", "Segoe UI", system-ui, sans-serif !important;
  color: #1e293b;
  box-sizing: border-box;
}

/* ── Card panels ──────────────────────────────────────────────────────── */
.ui-panel {
  background: #ffffff;
  border: 1px solid #e2e8f0;
  border-radius: 13px;
  padding: 12px 12px 10px 12px;
  box-shadow: 0 1px 3px rgba(0,0,0,0.05), 0 4px 16px rgba(0,0,0,0.04);
}

/* ── Section headers ─────────────────────────────────────────────────── */
.ui-section-header {
  margin: 0 0 10px 0;
  padding: 7px 12px 8px 12px;
  border-radius: 8px;
  background: linear-gradient(130deg, #0f766e 0%, #0d9488 55%, #14b8a6 100%);
}
.ui-section-title {
  color: #ffffff;
  font-size: 11px;
  font-weight: 800;
  letter-spacing: 0.09em;
  text-transform: uppercase;
}
.ui-section-subtitle {
  color: rgba(255,255,255,0.72);
  font-size: 10px;
  margin-top: 2px;
  font-weight: 400;
  letter-spacing: 0.01em;
}

/* ── Tabs ─────────────────────────────────────────────────────────────── */
/* Cover both old (p-) and new (lm-) JupyterLab class prefixes */
.ui-root .widget-tab > .p-TabBar,
.ui-root .widget-tab > .lm-TabBar {
  border-bottom: 2px solid #e2e8f0 !important;
  padding-left: 4px;
}
.ui-root .widget-tab > .p-TabBar .p-TabBar-tab,
.ui-root .widget-tab > .lm-TabBar .lm-TabBar-tab {
  background: #f1f5f9 !important;
  border: 1px solid #e2e8f0 !important;
  border-bottom: none !important;
  border-radius: 7px 7px 0 0 !important;
  color: #64748b !important;
  font-size: 11.5px !important;
  font-weight: 600 !important;
  padding: 5px 11px !important;
  margin-right: 3px;
  transition: background 120ms, color 120ms;
}
.ui-root .widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current,
.ui-root .widget-tab > .lm-TabBar .lm-TabBar-tab.lm-mod-current {
  background: #ffffff !important;
  color: #1e293b !important;
  border-bottom: 2px solid #ffffff !important;
  border-top: 2.5px solid #0d9488 !important;
}
.ui-root .widget-tab > .p-TabBar .p-TabBar-tab:not(.p-mod-current):hover,
.ui-root .widget-tab > .lm-TabBar .lm-TabBar-tab:not(.lm-mod-current):hover {
  background: #e2e8f0 !important;
  color: #334155 !important;
}
.ui-root .widget-tab > .p-TabPanel,
.ui-root .widget-tab > .lm-TabPanel {
  background: #ffffff !important;
  border: 1px solid #e2e8f0 !important;
  border-top: none !important;
  border-radius: 0 0 10px 10px !important;
  padding: 10px !important;
}

/* ── Text / textarea inputs ──────────────────────────────────────────── */
.ui-root .widget-text input,
.ui-root .widget-textarea textarea {
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 7px !important;
  background: #f8fafc !important;
  color: #1e293b !important;
  font-size: 12px;
  padding: 3px 8px !important;
  transition: border-color 120ms, box-shadow 120ms;
}
.ui-root .widget-text input:focus,
.ui-root .widget-textarea textarea:focus {
  border-color: #0d9488 !important;
  box-shadow: 0 0 0 2.5px rgba(13,148,136,0.20) !important;
  background: #fff !important;
  outline: none !important;
}

/* ── Dropdowns ───────────────────────────────────────────────────────── */
.ui-root .widget-dropdown select {
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 7px !important;
  background: #f8fafc !important;
  color: #1e293b !important;
  font-size: 12px;
  padding: 2px 6px !important;
}
.ui-root .widget-dropdown select:focus {
  border-color: #0d9488 !important;
  box-shadow: 0 0 0 2.5px rgba(13,148,136,0.20) !important;
  outline: none !important;
}

/* ── Number inputs ───────────────────────────────────────────────────── */
.ui-root .widget-numtext input,
.ui-root .widget-floattext input,
.ui-root .widget-boundedinttext input,
.ui-root .widget-boundedfloattext input {
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 7px !important;
  background: #f8fafc !important;
  font-size: 12px;
  padding: 3px 8px !important;
}
.ui-root .widget-numtext input:focus,
.ui-root .widget-floattext input:focus,
.ui-root .widget-boundedinttext input:focus,
.ui-root .widget-boundedfloattext input:focus {
  border-color: #0d9488 !important;
  box-shadow: 0 0 0 2.5px rgba(13,148,136,0.20) !important;
  outline: none !important;
}

/* ── SelectMultiple ──────────────────────────────────────────────────── */
.ui-root .widget-select-multiple select {
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 7px !important;
  background: #f8fafc !important;
  font-size: 12px;
}
.ui-root .widget-select-multiple select:focus {
  border-color: #0d9488 !important;
  box-shadow: 0 0 0 2.5px rgba(13,148,136,0.20) !important;
  outline: none !important;
}

/* ── Base button ─────────────────────────────────────────────────────── */
.ui-root .widget-button button {
  border-radius: 8px !important;
  font-size: 12.5px !important;
  font-weight: 600 !important;
  padding: 5px 12px !important;
  border: 1.5px solid #cbd5e1 !important;
  background: #f1f5f9 !important;
  color: #334155 !important;
  transition: filter 100ms, transform 100ms, box-shadow 100ms;
  cursor: pointer;
}
.ui-root .widget-button button:hover {
  filter: brightness(0.96);
  transform: translateY(-1px);
  box-shadow: 0 2px 8px rgba(0,0,0,0.10) !important;
}
.ui-root .widget-button button:active {
  transform: translateY(0) !important;
  filter: brightness(0.90) !important;
}

/* Plot (primary) */
.ui-root .ui-btn-primary button {
  background: linear-gradient(180deg, #3fa2a2 0%, #006464 100%) !important;
  color: #fff !important;
  border-color: #006464 !important;
  box-shadow: 0 1px 5px rgba(13,148,136,0.35) !important;
}
/* Save (secondary / purple) */
.ui-root .ui-btn-secondary button {
  background: linear-gradient(180deg, #7c3aed 0%, #6d28d9 100%) !important;
  color: #fff !important;
  border-color: #6d28d9 !important;
  box-shadow: 0 1px 5px rgba(124,58,237,0.28) !important;
}
/* Ghost (All / Clear / utility) */
.ui-root .ui-btn-ghost button {
  background: #ffffff !important;
  color: #475569 !important;
  border: 1.5px solid #cbd5e1 !important;
}
.ui-root .ui-btn-ghost button:hover {
  background: #f1f5f9 !important;
  border-color: #94a3b8 !important;
}

/* ── Toggle buttons ──────────────────────────────────────────────────── */
.ui-root .widget-toggle-buttons .widget-button button {
  border-radius: 0 !important;
  border-right-width: 0 !important;
  font-size: 12px !important;
  font-weight: 600 !important;
}
.ui-root .widget-toggle-buttons .widget-button:first-child button {
  border-radius: 7px 0 0 7px !important;
  border-left-width: 1.5px !important;
}
.ui-root .widget-toggle-buttons .widget-button:last-child button {
  border-radius: 0 7px 7px 0 !important;
  border-right-width: 1.5px !important;
}

/* ── Checkboxes ───────────────────────────────────────────────────────── */
.ui-root .widget-checkbox { margin: 2px 0; }
.ui-root .widget-checkbox input[type="checkbox"] { accent-color: #3fa2a2; }
.ui-root .widget-checkbox .widget-label {
  font-size: 12.5px !important;
  color: #334155 !important;
}

/* ── Description labels ──────────────────────────────────────────────── */
.ui-root .widget-label-basic,
.ui-root .widget-label {
  font-size: 12px;
  color: #475569;
  font-weight: 500;
}
.ui-root .widget-html-content { line-height: 1.4; }

/* ── Help-text pill ───────────────────────────────────────────────────── */
.ui-help {
  display: block;
  background: #f0fdfa;
  border: 1px solid #99f6e4;
  border-left: 3px solid #0d9488;
  border-radius: 6px;
  padding: 5px 10px;
  font-size: 11.5px;
  color: #0f766e;
  margin-bottom: 8px;
  line-height: 1.4;
}

/* ── Color pickers ────────────────────────────────────────────────────── */
.ui-root .widget-colorpicker input[type="color"] {
  border: 1.5px solid #cbd5e1 !important;
  border-radius: 5px;
  cursor: pointer;
  height: 26px;
}

/* ── Action bar ───────────────────────────────────────────────────────── */
.ui-action-bar {
  margin-top: 12px !important;
  padding: 8px 12px !important;
  background: #f8fafc !important;
  border: 1px solid #e2e8f0 !important;
  border-radius: 10px !important;
}
</style>
""")

# ── Widgets ───────────────────────────────────────────────────────────────────
_fig_name_wt = widgets.Text(value="", placeholder="auto", description="Filename:")

_run_mode_block = widgets.VBox([
    _section_header("Run Mode"),
    _run_mode,
])

_data_tabs = widgets.Tab(children=[_metrics_pane, _rdf_pane, _rms_pane])

_chart_type_row = widgets.VBox([_chart_type])

_section2_box = widgets.VBox([
    _section_header("Chart Options"),
    _chart_type_row,
    _chart_sub_box,
    _run_mode_block,
])

_style_theme_row = widgets.HBox([_theme_dd, _palette_dd])
_style_font_row  = widgets.HBox([_font_size, _font_fam])
_style_fig_row   = widgets.HBox([_fig_auto, _fig_w, _fig_h, _dpi_it])
_style_pane = widgets.VBox([_style_theme_row, _style_font_row, _style_fig_row], layout=_L_PANEL)
_axes_pane  = widgets.VBox([
    _title_wt, _xlabel_wt, _ylabel_wt,
    _use_xlim_cb, _xlim_box,
    _use_ylim_cb, _ylim_box,
], layout=_L_PANEL)
_appearance_tabs = widgets.Tab(children=[_style_pane, _axes_pane])

_action_bar = widgets.HBox(
    [_plot_btn, _save_btn, _fig_name_wt, _status],
    layout=widgets.Layout(gap="8px", align_items="center"),
)

# ── Column layout ─────────────────────────────────────────────────────────────
_left_col = widgets.VBox([
    _section_header("Systems", "Select systems to compare"),
    _sys_panel,
    _section_header("Series Colors"),
    _color_box,
], layout=_L_LEFT)

_mid_col = widgets.VBox([
    _section_header("Data", "Metrics · RDF · RMSD/RMSF · Scatter · Co-occurrence · Clustering"),
    _data_tabs,
    _section2_box,
], layout=_L_MID)

_main_body = widgets.HBox(
    [_left_col, _mid_col],
    layout=widgets.Layout(width=_FIXED_WIDTH, gap="10px"),
)

_bottom_col = widgets.VBox([
    _section_header("Appearance & Export"),
    _appearance_tabs,
    _action_bar,
], layout=widgets.Layout(width=_FIXED_WIDTH, padding="0 16px"))

# ── Assemble ───────────────────────────────────────────────────────────────────
MAIN_UI = widgets.VBox([
    _ui_css,
    _main_body,
    _bottom_col,
    _out,
], layout=widgets.Layout(width=_FIXED_WIDTH))

MAIN_UI.add_class("ui-root")
for _p in (_left_col, _mid_col, _bottom_col, _section2_box):
    _p.add_class("ui-panel")

# Buttons
_plot_btn.add_class("ui-btn-primary")
_save_btn.add_class("ui-btn-secondary")
_sys_btn_all.add_class("ui-btn-ghost")
_sys_btn_none.add_class("ui-btn-ghost")
_plot_btn.layout.width = "90px"
_save_btn.layout.width = "90px"
_status.layout.min_width = "220px"

# Action bar styling
_action_bar.add_class("ui-action-bar")

# Update help text widgets to use the styled pill class
_metrics_help.value = "<div class='ui-help'>Choose a metric type and metric, then plot averaged or individual runs.</div>"
_rdf_help.value     = "<div class='ui-help'>Choose center / type, optionally select residues, then plot RDF curves.</div>"
_rms_help.value     = "<div class='ui-help'>Choose RMSD or RMSF and a metric, then pick bar / KDE / time-series.</div>"


# ── Dispatchers ────────────────────────────────────────────────────────────────
def _run_plot_dispatch(*_):
    idx = _data_tabs.selected_index
    handler = {
        1: _run_rdf_plot,
        2: _run_rms_plot,
        3: _run_scatter_plot,
        4: _run_cooccurrence_plot,
        5: _run_clustering_plot,
    }.get(idx, _run_plot)
    handler()


def _plot_filename_stem(idx: int) -> str:
    if idx == 1:
        center = str(_rdf_center_dd.value or "rdf").lower().replace(" ", "_")
        rtype  = str(_rdf_type_dd.value or "").lower().replace(" ", "_")
        return f"rdf_{center}_{rtype}"
    if idx == 2:
        kind   = str(_rms_kind_dd.value or "rms").lower()
        metric = str(_rms_metric_dd.value or "").lower().replace(" ", "_")
        chart  = _chart_type.value.lower().replace(" ", "_").replace("%", "pct")
        return f"{kind}_{chart}_{metric}"
    if idx == 3:
        return f"scatter_{_scatter_x_dd.value or 'x_metric'}_vs_{_scatter_y_dd.value or 'y_metric'}"
    if idx == 4:
        return f"cooccurrence_{_cooccurrence_slug()}"
    if idx == 5:
        cfg_name = (
            _CLUSTER_CATALOG[_cluster_config_dd.value]["config_name"]
            if _CLUSTER_CATALOG and _cluster_config_dd.value is not None else "clustering"
        )
        return f"clustering_{cfg_name}_{_cluster_group_by.value}"
    chart = str(_chart_type.value).lower().replace(" ", "_").replace("%", "pct")
    return f"metrics_{chart}_{_metric_dd.value or 'metric'}"


def _safe_custom_figure_name() -> str | None:
    custom_name = str(_fig_name_wt.value or "").strip()
    if not custom_name:
        return None
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", custom_name).strip("._")
    return safe_name or None


def _save_plot(*_):
    if _last_fig is None:
        _status.value = "<span style='color:#dc2626;font-weight:600'>Nothing to save — plot first.</span>"
        return
    out_dir = Path("./Comparison_Plots")
    out_dir.mkdir(parents=True, exist_ok=True)
    idx   = _data_tabs.selected_index
    fname = out_dir / f"{_safe_custom_figure_name() or _plot_filename_stem(idx)}.png"
    _last_fig.savefig(fname, dpi=_dpi_it.value, bbox_inches="tight")
    _status.value = (
        f"<span style='color:#16a34a;font-weight:600'>"
        f"&#10003; Saved: {fname.name}"
        f"</span>"
    )


def _reset_colors_to_palette(*_):
    _UNIT_COLOR_MAP.clear()
    _rebuild_color_pickers()


def _maybe_get_callable(name: str):
    fn = globals().get(name)
    return fn if callable(fn) else None


def _observe_call(widget, fn_name: str, names: str = "value") -> None:
    fn = _maybe_get_callable(fn_name)
    if fn is not None:
        widget.observe(fn, names=names)


def _observe_trigger(widget, fn_name: str, names: str = "value") -> None:
    fn = _maybe_get_callable(fn_name)
    if fn is not None:
        widget.observe(lambda _change: fn(), names=names)


def _call_if_available(fn_name: str) -> None:
    fn = _maybe_get_callable(fn_name)
    if fn is not None:
        fn()


# ── Append scatter / co-occurrence / clustering panes ──────────────────────────
for _pane_name in ("_scatter_pane", "_cooccurrence_pane", "_clustering_pane"):
    if _pane_name in globals():
        _pane_obj = globals()[_pane_name]
        if _pane_obj not in _data_tabs.children:
            _data_tabs.children = tuple(list(_data_tabs.children) + [_pane_obj])

for _i, _lbl in enumerate(["Metrics", "RDF", "RMSD / RMSF", "2D Scatter", "Co-occurrence", "Clustering"]):
    if _i < len(_data_tabs.children):
        _data_tabs.set_title(_i, _lbl)

if _DEFAULT_DATA_TAB_INDEX < len(_data_tabs.children):
    _data_tabs.selected_index = _DEFAULT_DATA_TAB_INDEX

for _i, _lbl in enumerate(["Style", "Axes / Labels"]):
    _appearance_tabs.set_title(_i, _lbl)

# ── Wire observers ─────────────────────────────────────────────────────────────
for _cb in _sys_checks.values():
    _cb.observe(_on_systems_change, names="value")
    _observe_trigger(_cb, "_update_scatter_metric_options")
    _observe_trigger(_cb, "_update_cooccurrence_metric_options")

for _widget in (_contact_filter_cb, _contact_cutoff_wt, _contact_min_pct_wt):
    _widget.observe(_contact_filter_changed, names="value")

_observe_call(_scatter_x_search_wt, "_update_scatter_metric_options")
_observe_call(_scatter_y_search_wt, "_update_scatter_metric_options")
_observe_call(_scatter_x_dd,        "_update_scatter_info")
_observe_call(_scatter_y_dd,        "_update_scatter_info")
_observe_call(_scatter_mode,        "_update_scatter_info")

_call_if_available("_update_scatter_metric_options")
_call_if_available("_update_cooccurrence_metric_options")
_call_if_available("_refresh_clustering_options")

_plot_btn.on_click(_run_plot_dispatch)
_save_btn.on_click(_save_plot)
_sys_search.observe(_filter_sys,                         names="value")
_use_xlim_cb.observe(_sync_axis_boxes,                   names="value")
_use_ylim_cb.observe(_sync_axis_boxes,                   names="value")
_show_thresh_cb.observe(_rebuild_chart_sub_options,      names="value")
_rdf_thresholds_cb.observe(_rebuild_chart_sub_options,   names="value")
_rms_thresholds_cb.observe(_rebuild_chart_sub_options,   names="value")
_scatter_thresholds_x_cb.observe(_rebuild_chart_sub_options, names="value")
_scatter_thresholds_y_cb.observe(_rebuild_chart_sub_options, names="value")
_chart_type.observe(_rebuild_chart_sub_options,          names="value")
_data_tabs.observe(_update_chart_type_options,           names="selected_index")
_rms_kind_dd.observe(_update_chart_type_options,         names="value")
_rdf_center_dd.observe(_update_rdf_type_options,         names="value")
_rdf_type_dd.observe(_update_rdf_detail_options,         names="value")
_rdf_view.observe(_update_rdf_detail_options,            names="value")
_fig_auto.observe(_sync_fig_boxes,                       names="value")
_rms_kind_dd.observe(_update_rms_metric_options,         names="value")
_palette_dd.observe(_reset_colors_to_palette,            names="value")

_install_contact_map_controls()
_install_rms_systems()
_sync_fig_boxes()
_rebuild_color_pickers()
_update_metric_options()
_update_chart_type_options()
_rebuild_chart_sub_options()
_refresh_rdf_options()
_refresh_rms_options()


In [21]:
display(MAIN_UI)
